In [2]:
import numpy as np
import pandas as pd

In [3]:
data_claim = pd.read_csv('data/Data_Klaim.csv')
data_polis = pd.read_csv('data/Data_Polis.csv')

In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# >>> THE MAXIMUM STABILITY UPGRADE: LOG TRANSFORM <<<
weekly_df['Frequency_Log'] = np.log1p(weekly_df['Frequency'])
weekly_df['Total_Claim_Log'] = np.log1p(weekly_df['Total_Claim'])

# ==========================================
# 2. DLINEAR DECOMPOSITION (ON LOG DATA)
# ==========================================
print("2. Performing DLinear Decomposition on Log-Scaled Data...")

window_size = 4 

# We decompose the LOGGED features, not the raw billions
targets = ['Frequency_Log', 'Total_Claim_Log']

for col in targets:
    weekly_df[f'{col}_Trend'] = weekly_df[col].rolling(window=window_size).mean()
    weekly_df[f'{col}_Remain'] = weekly_df[col] - weekly_df[f'{col}_Trend']
    
    for lag in range(1, 5):
        weekly_df[f'{col}_Trend_Lag{lag}'] = weekly_df[f'{col}_Trend'].shift(lag)
        weekly_df[f'{col}_Remain_Lag{lag}'] = weekly_df[f'{col}_Remain'].shift(lag)

train_df = weekly_df.dropna().copy()

# ==========================================
# 3. TRAIN DLINEAR LINEAR LAYERS
# ==========================================
print("3. Training Dual Linear Layers in Log Space...")

model_params = {'alpha': 1.0}

# --- FREQUENCY MODELS ---
features_freq_trend = [f'Frequency_Log_Trend_Lag{i}' for i in range(1, 5)]
model_freq_trend = Ridge(**model_params)
model_freq_trend.fit(train_df[features_freq_trend], train_df['Frequency_Log_Trend'])

features_freq_remain = [f'Frequency_Log_Remain_Lag{i}' for i in range(1, 5)]
model_freq_remain = Ridge(**model_params)
model_freq_remain.fit(train_df[features_freq_remain], train_df['Frequency_Log_Remain'])

# --- TOTAL CLAIM MODELS ---
features_tot_trend = [f'Total_Claim_Log_Trend_Lag{i}' for i in range(1, 5)]
model_tot_trend = Ridge(**model_params)
model_tot_trend.fit(train_df[features_tot_trend], train_df['Total_Claim_Log_Trend'])

features_tot_remain = [f'Total_Claim_Log_Remain_Lag{i}' for i in range(1, 5)]
model_tot_remain = Ridge(**model_params)
model_tot_remain.fit(train_df[features_tot_remain], train_df['Total_Claim_Log_Remain'])

# ==========================================
# 4. RECURSIVE WEEKLY FORECASTING
# ==========================================
print("4. Forecasting Future Weeks recursively...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')
forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')

current_history = weekly_df.copy()
weekly_predictions = []

for week_end in forecast_weeks:
    last_4 = current_history.tail(4)
    
    # --- FREQUENCY PREDICTION ---
    row_freq_trend = pd.DataFrame([{f'Frequency_Log_Trend_Lag{i}': last_4['Frequency_Log_Trend'].iloc[-i] for i in range(1, 5)}])
    row_freq_remain = pd.DataFrame([{f'Frequency_Log_Remain_Lag{i}': last_4['Frequency_Log_Remain'].iloc[-i] for i in range(1, 5)}])
    
    pred_f_trend_log = model_freq_trend.predict(row_freq_trend)[0]
    pred_f_remain_log = model_freq_remain.predict(row_freq_remain)[0]
    
    pred_freq_log = pred_f_trend_log + pred_f_remain_log
    pred_freq = np.expm1(pred_freq_log) # <--- CONVERT BACK TO REAL NUMBERS
    pred_freq = max(0, pred_freq) 
    
    # --- TOTAL CLAIM PREDICTION ---
    row_tot_trend = pd.DataFrame([{f'Total_Claim_Log_Trend_Lag{i}': last_4['Total_Claim_Log_Trend'].iloc[-i] for i in range(1, 5)}])
    row_tot_remain = pd.DataFrame([{f'Total_Claim_Log_Remain_Lag{i}': last_4['Total_Claim_Log_Remain'].iloc[-i] for i in range(1, 5)}])
    
    pred_t_trend_log = model_tot_trend.predict(row_tot_trend)[0]
    pred_t_remain_log = model_tot_remain.predict(row_tot_remain)[0]
    
    pred_total_log = pred_t_trend_log + pred_t_remain_log
    pred_total = np.expm1(pred_total_log) # <--- CONVERT BACK TO RUPIAH
    pred_total = max(0, pred_total)
    
    # Store Prediction
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })
    
    # Update History for the Recursive Loop
    new_row_data = {
        'Week_End_Date': week_end, 
        'Frequency': pred_freq, 'Total_Claim': pred_total,
        'Frequency_Log': pred_freq_log, 'Total_Claim_Log': pred_total_log
    }
    
    hist_plus_new = pd.concat([current_history, pd.DataFrame([new_row_data])], ignore_index=True)
    
    for col in targets:
        hist_plus_new[f'{col}_Trend'] = hist_plus_new[col].rolling(window=window_size).mean()
        hist_plus_new[f'{col}_Remain'] = hist_plus_new[col] - hist_plus_new[f'{col}_Trend']
        
    current_history = hist_plus_new.copy()

# ==========================================
# 5. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("5. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 6. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_dlinear_logscaled.csv', index=False)

print("\n--- FINAL FORECAST (Log-Scaled DLinear Architecture) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_dlinear_logscaled.csv' saved! Safe from mathematical explosions!")

1. Processing Raw Data to Weekly Level...
2. Performing DLinear Decomposition on Log-Scaled Data...
3. Training Dual Linear Layers in Log Space...
4. Forecasting Future Weeks recursively...
5. Apportioning to Daily and Rolling up to Exact Calendar Months...

--- FINAL FORECAST (Log-Scaled DLinear Architecture) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        218  4.871453e+07  1.061977e+10
1      2025-09        232  5.006399e+07  1.161485e+10
2      2025-10        241  5.052364e+07  1.217620e+10
3      2025-11        234  5.061158e+07  1.184311e+10
4      2025-12        242  5.065958e+07  1.225962e+10

File 'submission_dlinear_logscaled.csv' saved! Safe from mathematical explosions!


DLinear w/ Auto-Tuned & Cyclic

In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# The Stability Upgrade: Log Transform
weekly_df['Frequency_Log'] = np.log1p(weekly_df['Frequency'])
weekly_df['Total_Claim_Log'] = np.log1p(weekly_df['Total_Claim'])

# ==========================================
# 2. CYCLIC CALENDAR & DLINEAR DECOMPOSITION
# ==========================================
print("2. Performing Decomposition and Cyclic Engineering...")

# Injecting Calendar Waves
weekly_df['Week_Num'] = weekly_df['Week_End_Date'].dt.isocalendar().week.astype(int)
weekly_df['week_sin'] = np.sin(2 * np.pi * weekly_df['Week_Num'] / 52.0)
weekly_df['week_cos'] = np.cos(2 * np.pi * weekly_df['Week_Num'] / 52.0)

window_size = 4 
lookback = 6 # Upgraded from 4 to 6 weeks of safe memory

targets = ['Frequency_Log', 'Total_Claim_Log']

for col in targets:
    weekly_df[f'{col}_Trend'] = weekly_df[col].rolling(window=window_size).mean()
    weekly_df[f'{col}_Remain'] = weekly_df[col] - weekly_df[f'{col}_Trend']
    
    for lag in range(1, lookback + 1):
        weekly_df[f'{col}_Trend_Lag{lag}'] = weekly_df[f'{col}_Trend'].shift(lag)
        weekly_df[f'{col}_Remain_Lag{lag}'] = weekly_df[f'{col}_Remain'].shift(lag)

train_df = weekly_df.dropna().copy()

# ==========================================
# 3. TRAIN AUTO-TUNING LINEAR LAYERS
# ==========================================
print("3. Training Auto-Tuned Dual Linear Layers...")

# Let the machine find the perfect mathematical penalty
alphas_to_test = [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]

# --- FREQUENCY MODELS ---
# Trend only looks at pure momentum
features_freq_trend = [f'Frequency_Log_Trend_Lag{i}' for i in range(1, lookback + 1)]
model_freq_trend = RidgeCV(alphas=alphas_to_test)
model_freq_trend.fit(train_df[features_freq_trend], train_df['Frequency_Log_Trend'])

# Remainder looks at momentum PLUS the calendar seasonality
features_freq_remain = [f'Frequency_Log_Remain_Lag{i}' for i in range(1, lookback + 1)] + ['week_sin', 'week_cos']
model_freq_remain = RidgeCV(alphas=alphas_to_test)
model_freq_remain.fit(train_df[features_freq_remain], train_df['Frequency_Log_Remain'])

# --- TOTAL CLAIM MODELS ---
features_tot_trend = [f'Total_Claim_Log_Trend_Lag{i}' for i in range(1, lookback + 1)]
model_tot_trend = RidgeCV(alphas=alphas_to_test)
model_tot_trend.fit(train_df[features_tot_trend], train_df['Total_Claim_Log_Trend'])

features_tot_remain = [f'Total_Claim_Log_Remain_Lag{i}' for i in range(1, lookback + 1)] + ['week_sin', 'week_cos']
model_tot_remain = RidgeCV(alphas=alphas_to_test)
model_tot_remain.fit(train_df[features_tot_remain], train_df['Total_Claim_Log_Remain'])

print(f"Optimal Trend Alpha (Tot): {model_tot_trend.alpha_}")
print(f"Optimal Remain Alpha (Tot): {model_tot_remain.alpha_}")

# ==========================================
# 4. RECURSIVE WEEKLY FORECASTING
# ==========================================
print("4. Forecasting Future Weeks recursively...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')
forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')

current_history = weekly_df.copy()
weekly_predictions = []

for week_end in forecast_weeks:
    last_window = current_history.tail(lookback)
    
    # Calculate current future calendar wave
    week_num = week_end.isocalendar().week
    current_sin = np.sin(2 * np.pi * week_num / 52.0)
    current_cos = np.cos(2 * np.pi * week_num / 52.0)
    
    # --- FREQUENCY PREDICTION ---
    dict_freq_trend = {f'Frequency_Log_Trend_Lag{i}': last_window['Frequency_Log_Trend'].iloc[-i] for i in range(1, lookback + 1)}
    row_freq_trend = pd.DataFrame([dict_freq_trend])[features_freq_trend]
    
    dict_freq_remain = {f'Frequency_Log_Remain_Lag{i}': last_window['Frequency_Log_Remain'].iloc[-i] for i in range(1, lookback + 1)}
    dict_freq_remain['week_sin'] = current_sin
    dict_freq_remain['week_cos'] = current_cos
    row_freq_remain = pd.DataFrame([dict_freq_remain])[features_freq_remain]
    
    pred_f_trend_log = model_freq_trend.predict(row_freq_trend)[0]
    pred_f_remain_log = model_freq_remain.predict(row_freq_remain)[0]
    
    pred_freq_log = pred_f_trend_log + pred_f_remain_log
    pred_freq = np.expm1(pred_freq_log) 
    pred_freq = max(0, pred_freq) 
    
    # --- TOTAL CLAIM PREDICTION ---
    dict_tot_trend = {f'Total_Claim_Log_Trend_Lag{i}': last_window['Total_Claim_Log_Trend'].iloc[-i] for i in range(1, lookback + 1)}
    row_tot_trend = pd.DataFrame([dict_tot_trend])[features_tot_trend]
    
    dict_tot_remain = {f'Total_Claim_Log_Remain_Lag{i}': last_window['Total_Claim_Log_Remain'].iloc[-i] for i in range(1, lookback + 1)}
    dict_tot_remain['week_sin'] = current_sin
    dict_tot_remain['week_cos'] = current_cos
    row_tot_remain = pd.DataFrame([dict_tot_remain])[features_tot_remain]
    
    pred_t_trend_log = model_tot_trend.predict(row_tot_trend)[0]
    pred_t_remain_log = model_tot_remain.predict(row_tot_remain)[0]
    
    pred_total_log = pred_t_trend_log + pred_t_remain_log
    pred_total = np.expm1(pred_total_log) 
    pred_total = max(0, pred_total)
    
    # Store Prediction
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })
    
    # Update History
    new_row_data = {
        'Week_End_Date': week_end, 
        'Frequency': pred_freq, 'Total_Claim': pred_total,
        'Frequency_Log': pred_freq_log, 'Total_Claim_Log': pred_total_log,
        'Week_Num': week_num, 'week_sin': current_sin, 'week_cos': current_cos
    }
    
    hist_plus_new = pd.concat([current_history, pd.DataFrame([new_row_data])], ignore_index=True)
    
    for col in targets:
        hist_plus_new[f'{col}_Trend'] = hist_plus_new[col].rolling(window=window_size).mean()
        hist_plus_new[f'{col}_Remain'] = hist_plus_new[col] - hist_plus_new[f'{col}_Trend']
        
    current_history = hist_plus_new.copy()

# ==========================================
# 5. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("5. Apportioning to Daily and Monthly Roll-up...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']
final_forecast['Severity'] = final_forecast['Total_Claim'] / final_forecast['Frequency']

# ==========================================
# 6. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_dlinear_log_v2.csv', index=False)

print("\n--- FINAL FORECAST (V2: Log-Scaled, Auto-Tuned, Cyclic DLinear) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_dlinear_log_v2.csv' saved!")

1. Processing Raw Data to Weekly Level...
2. Performing Decomposition and Cyclic Engineering...
3. Training Auto-Tuned Dual Linear Layers...
Optimal Trend Alpha (Tot): 0.1
Optimal Remain Alpha (Tot): 100.0
4. Forecasting Future Weeks recursively...
5. Apportioning to Daily and Monthly Roll-up...

--- FINAL FORECAST (V2: Log-Scaled, Auto-Tuned, Cyclic DLinear) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        208  4.578487e+07  9.523252e+09
1      2025-09        226  4.822671e+07  1.089924e+10
2      2025-10        236  4.923699e+07  1.161993e+10
3      2025-11        229  4.932857e+07  1.129624e+10
4      2025-12        237  4.934223e+07  1.169411e+10

File 'submission_dlinear_log_v2.csv' saved!


In [6]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

data_claim = pd.read_csv('data/Data_Klaim.csv')
data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])

valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# >>> THE MAXIMUM STABILITY UPGRADE: LOG TRANSFORM <<<
weekly_df['Frequency_Log'] = np.log1p(weekly_df['Frequency'])
weekly_df['Total_Claim_Log'] = np.log1p(weekly_df['Total_Claim'])

# ==========================================
# 2. ADVANCED DECOMPOSITION (EMA ON LOG DATA)
# ==========================================
print("2. Performing EMA DLinear Decomposition...")

window_size = 4 
lookback = 4
targets = ['Frequency_Log', 'Total_Claim_Log']

for col in targets:
    # UPGRADE 1: Exponential Moving Average (EMA) instead of Simple Rolling Mean
    weekly_df[f'{col}_Trend'] = weekly_df[col].ewm(span=window_size, adjust=False).mean()
    weekly_df[f'{col}_Remain'] = weekly_df[col] - weekly_df[f'{col}_Trend']
    
    for lag in range(1, lookback + 1):
        weekly_df[f'{col}_Trend_Lag{lag}'] = weekly_df[f'{col}_Trend'].shift(lag)
        weekly_df[f'{col}_Remain_Lag{lag}'] = weekly_df[f'{col}_Remain'].shift(lag)

train_df = weekly_df.dropna().copy()

# ==========================================
# 3. TRAIN DLINEAR WITH AUTOTUNED RIDGE (RidgeCV)
# ==========================================
print("3. Training Dual Linear Layers with Auto-Tuned L2 Regularization...")

# UPGRADE 2: RidgeCV hunts for the best alpha to prevent overfitting on small data
cv_alphas = (0.1, 1.0, 5.0, 10.0)

# --- FREQUENCY MODELS ---
features_freq_trend = [f'Frequency_Log_Trend_Lag{i}' for i in range(1, lookback + 1)]
model_freq_trend = RidgeCV(alphas=cv_alphas)
model_freq_trend.fit(train_df[features_freq_trend], train_df['Frequency_Log_Trend'])

features_freq_remain = [f'Frequency_Log_Remain_Lag{i}' for i in range(1, lookback + 1)]
model_freq_remain = RidgeCV(alphas=cv_alphas)
model_freq_remain.fit(train_df[features_freq_remain], train_df['Frequency_Log_Remain'])

# --- TOTAL CLAIM MODELS ---
features_tot_trend = [f'Total_Claim_Log_Trend_Lag{i}' for i in range(1, lookback + 1)]
model_tot_trend = RidgeCV(alphas=cv_alphas)
model_tot_trend.fit(train_df[features_tot_trend], train_df['Total_Claim_Log_Trend'])

features_tot_remain = [f'Total_Claim_Log_Remain_Lag{i}' for i in range(1, lookback + 1)]
model_tot_remain = RidgeCV(alphas=cv_alphas)
model_tot_remain.fit(train_df[features_tot_remain], train_df['Total_Claim_Log_Remain'])

# ==========================================
# 4. RECURSIVE WEEKLY FORECASTING
# ==========================================
print("4. Forecasting Future Weeks recursively...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')
forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')

current_history = weekly_df.copy()
weekly_predictions = []

for week_end in forecast_weeks:
    last_n = current_history.tail(lookback)
    
    # --- FREQUENCY PREDICTION ---
    row_freq_trend = pd.DataFrame([{f'Frequency_Log_Trend_Lag{i}': last_n['Frequency_Log_Trend'].iloc[-i] for i in range(1, lookback + 1)}])
    row_freq_remain = pd.DataFrame([{f'Frequency_Log_Remain_Lag{i}': last_n['Frequency_Log_Remain'].iloc[-i] for i in range(1, lookback + 1)}])
    
    pred_f_trend_log = model_freq_trend.predict(row_freq_trend)[0]
    pred_f_remain_log = model_freq_remain.predict(row_freq_remain)[0]
    
    pred_freq_log = pred_f_trend_log + pred_f_remain_log
    pred_freq = max(0, np.expm1(pred_freq_log)) 
    
    # --- TOTAL CLAIM PREDICTION ---
    row_tot_trend = pd.DataFrame([{f'Total_Claim_Log_Trend_Lag{i}': last_n['Total_Claim_Log_Trend'].iloc[-i] for i in range(1, lookback + 1)}])
    row_tot_remain = pd.DataFrame([{f'Total_Claim_Log_Remain_Lag{i}': last_n['Total_Claim_Log_Remain'].iloc[-i] for i in range(1, lookback + 1)}])
    
    pred_t_trend_log = model_tot_trend.predict(row_tot_trend)[0]
    pred_t_remain_log = model_tot_remain.predict(row_tot_remain)[0]
    
    pred_total_log = pred_t_trend_log + pred_t_remain_log
    pred_total = max(0, np.expm1(pred_total_log)) 
    
    # Store Prediction
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': pred_freq,
        'Total_Claim': pred_total
    })
    
    # Update History for the Recursive Loop
    new_row_data = {
        'Week_End_Date': week_end, 
        'Frequency': pred_freq, 'Total_Claim': pred_total,
        'Frequency_Log': pred_freq_log, 'Total_Claim_Log': pred_total_log
    }
    
    hist_plus_new = pd.concat([current_history, pd.DataFrame([new_row_data])], ignore_index=True)
    
    for col in targets:
        hist_plus_new[f'{col}_Trend'] = hist_plus_new[col].ewm(span=window_size, adjust=False).mean()
        hist_plus_new[f'{col}_Remain'] = hist_plus_new[col] - hist_plus_new[f'{col}_Trend']
        
    current_history = hist_plus_new.copy()

# ==========================================
# 5. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("5. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']

# UPGRADE 3: Safe Severity Calculation
final_forecast['Severity'] = np.where(final_forecast['Frequency'] > 0, 
                                      final_forecast['Total_Claim'] / final_forecast['Frequency'], 
                                      0)

# ==========================================
# 6. EXPORT
# ==========================================
formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_dlinear_ema_logscaled.csv', index=False)

print("\n--- FINAL FORECAST (EMA Log-Scaled DLinear Architecture) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_dlinear_ema_logscaled.csv' saved!")

1. Processing Raw Data to Weekly Level...
2. Performing EMA DLinear Decomposition...
3. Training Dual Linear Layers with Auto-Tuned L2 Regularization...
4. Forecasting Future Weeks recursively...
5. Apportioning to Daily and Rolling up to Exact Calendar Months...

--- FINAL FORECAST (EMA Log-Scaled DLinear Architecture) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        208  5.140748e+07  1.069276e+10
1      2025-09        229  5.036018e+07  1.153248e+10
2      2025-10        237  5.071076e+07  1.201845e+10
3      2025-11        230  5.072312e+07  1.166632e+10
4      2025-12        238  5.070758e+07  1.206840e+10

File 'submission_dlinear_ema_logscaled.csv' saved!


In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO WEEKLY AGGREGATION
# ==========================================
print("1. Processing Raw Data to Weekly Level...")

data_claim = pd.read_csv('data/Data_Klaim.csv')
data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])

valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# The Stability Upgrade: Log Transform for DLinear
weekly_df['Frequency_Log'] = np.log1p(weekly_df['Frequency'])
weekly_df['Total_Claim_Log'] = np.log1p(weekly_df['Total_Claim'])

# ==========================================
# 2. FEATURE ENGINEERING (DLINEAR + LIGHTGBM)
# ==========================================
print("2. Generating Features for DLinear (EMA) and LightGBM (Raw Lags)...")

window_size = 4 
lookback = 4
targets_log = ['Frequency_Log', 'Total_Claim_Log']
targets_raw = ['Frequency', 'Total_Claim']

# A. Generate EMA Decomposed Features for DLinear
for col in targets_log:
    weekly_df[f'{col}_Trend'] = weekly_df[col].ewm(span=window_size, adjust=False).mean()
    weekly_df[f'{col}_Remain'] = weekly_df[col] - weekly_df[f'{col}_Trend']
    
    for lag in range(1, lookback + 1):
        weekly_df[f'{col}_Trend_Lag{lag}'] = weekly_df[f'{col}_Trend'].shift(lag)
        weekly_df[f'{col}_Remain_Lag{lag}'] = weekly_df[f'{col}_Remain'].shift(lag)

# B. Generate Raw Lags for LightGBM
for col in targets_raw:
    for lag in range(1, lookback + 1):
        weekly_df[f'{col}_Lag{lag}'] = weekly_df[col].shift(lag)

train_df = weekly_df.dropna().copy()

# ==========================================
# 3. TRAIN BOTH ARCHITECTURES
# ==========================================
print("3. Training Ensembled Models...")

# --- 3A. TRAIN DLINEAR (RidgeCV) ---
cv_alphas = (0.1, 1.0, 5.0, 10.0)

# Frequency DLinear
features_freq_trend = [f'Frequency_Log_Trend_Lag{i}' for i in range(1, lookback + 1)]
features_freq_remain = [f'Frequency_Log_Remain_Lag{i}' for i in range(1, lookback + 1)]
dl_freq_trend = RidgeCV(alphas=cv_alphas).fit(train_df[features_freq_trend], train_df['Frequency_Log_Trend'])
dl_freq_remain = RidgeCV(alphas=cv_alphas).fit(train_df[features_freq_remain], train_df['Frequency_Log_Remain'])

# Total Claim DLinear
features_tot_trend = [f'Total_Claim_Log_Trend_Lag{i}' for i in range(1, lookback + 1)]
features_tot_remain = [f'Total_Claim_Log_Remain_Lag{i}' for i in range(1, lookback + 1)]
dl_tot_trend = RidgeCV(alphas=cv_alphas).fit(train_df[features_tot_trend], train_df['Total_Claim_Log_Trend'])
dl_tot_remain = RidgeCV(alphas=cv_alphas).fit(train_df[features_tot_remain], train_df['Total_Claim_Log_Remain'])

# --- 3B. TRAIN LIGHTGBM ---
# Tuned for weekly dataset (~75 rows): Small num_leaves and min_child_samples
lgb_params = {'n_estimators': 100, 'num_leaves': 10, 'min_child_samples': 4, 'random_state': 42, 'verbose': -1}

features_lgb_freq = [f'Frequency_Lag{i}' for i in range(1, lookback + 1)]
model_lgb_freq = lgb.LGBMRegressor(**lgb_params)
model_lgb_freq.fit(train_df[features_lgb_freq], train_df['Frequency'])

features_lgb_tot = [f'Total_Claim_Lag{i}' for i in range(1, lookback + 1)]
model_lgb_tot = lgb.LGBMRegressor(**lgb_params)
model_lgb_tot.fit(train_df[features_lgb_tot], train_df['Total_Claim'])

# ==========================================
# 4. RECURSIVE WEEKLY FORECASTING & BLENDING
# ==========================================
print("4. Executing Recursive Forecast & Golden Blend (75% DL / 25% LGBM)...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_date = pd.to_datetime('2025-12-31')
forecast_weeks = pd.date_range(start=last_hist_date + pd.Timedelta(days=7), 
                               end=target_end_date + pd.Timedelta(days=7), freq='W-SUN')

current_history = weekly_df.copy()
weekly_predictions = []

for week_end in forecast_weeks:
    last_n = current_history.tail(lookback)
    
    # --- A. DLINEAR PREDICTIONS ---
    r_freq_t = pd.DataFrame([{f'Frequency_Log_Trend_Lag{i}': last_n['Frequency_Log_Trend'].iloc[-i] for i in range(1, lookback + 1)}])
    r_freq_r = pd.DataFrame([{f'Frequency_Log_Remain_Lag{i}': last_n['Frequency_Log_Remain'].iloc[-i] for i in range(1, lookback + 1)}])
    pred_dl_freq = max(0, np.expm1(dl_freq_trend.predict(r_freq_t)[0] + dl_freq_remain.predict(r_freq_r)[0]))
    
    r_tot_t = pd.DataFrame([{f'Total_Claim_Log_Trend_Lag{i}': last_n['Total_Claim_Log_Trend'].iloc[-i] for i in range(1, lookback + 1)}])
    r_tot_r = pd.DataFrame([{f'Total_Claim_Log_Remain_Lag{i}': last_n['Total_Claim_Log_Remain'].iloc[-i] for i in range(1, lookback + 1)}])
    pred_dl_tot = max(0, np.expm1(dl_tot_trend.predict(r_tot_t)[0] + dl_tot_remain.predict(r_tot_r)[0]))

    # --- B. LIGHTGBM PREDICTIONS ---
    r_lgb_freq = pd.DataFrame([{f'Frequency_Lag{i}': last_n['Frequency'].iloc[-i] for i in range(1, lookback + 1)}])
    pred_lgb_freq = max(0, model_lgb_freq.predict(r_lgb_freq)[0])
    
    r_lgb_tot = pd.DataFrame([{f'Total_Claim_Lag{i}': last_n['Total_Claim'].iloc[-i] for i in range(1, lookback + 1)}])
    pred_lgb_tot = max(0, model_lgb_tot.predict(r_lgb_tot)[0])

    # --- C. THE GOLDEN BLEND (ENSEMBLE) ---
    final_pred_freq = (pred_dl_freq * 0.75) + (pred_lgb_freq * 0.25)
    final_pred_tot = (pred_dl_tot * 0.75) + (pred_lgb_tot * 0.25)
    
    # Store Final Prediction
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date': week_end,
        'Frequency': final_pred_freq,
        'Total_Claim': final_pred_tot
    })
    
    # Update History for the Recursive Loop using the BLENDED predictions
    new_row_data = {
        'Week_End_Date': week_end, 
        'Frequency': final_pred_freq, 
        'Total_Claim': final_pred_tot,
        'Frequency_Log': np.log1p(final_pred_freq), 
        'Total_Claim_Log': np.log1p(final_pred_tot)
    }
    
    hist_plus_new = pd.concat([current_history, pd.DataFrame([new_row_data])], ignore_index=True)
    
    # Recalculate EMA Trend and Remainder dynamically
    for col in targets_log:
        hist_plus_new[f'{col}_Trend'] = hist_plus_new[col].ewm(span=window_size, adjust=False).mean()
        hist_plus_new[f'{col}_Remain'] = hist_plus_new[col] - hist_plus_new[f'{col}_Trend']
        
    current_history = hist_plus_new.copy()

# ==========================================
# 5. DAILY APPORTIONMENT & MONTHLY ROLL-UP
# ==========================================
print("5. Apportioning to Daily and Rolling up to Exact Calendar Months...")

daily_records = []
for row in weekly_predictions:
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    daily_freq = row['Frequency'] / 7.0
    daily_total = row['Total_Claim'] / 7.0
    for d in days:
        daily_records.append({'Date': d, 'Daily_Freq': daily_freq, 'Daily_Total': daily_total})

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

cols_to_sum = ['Daily_Freq', 'Daily_Total']
monthly_forecast = daily_df.groupby('Month_Period')[cols_to_sum].sum().reset_index()

target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
final_forecast = monthly_forecast[(monthly_forecast['Month_Period'] >= target_start) & 
                                  (monthly_forecast['Month_Period'] <= target_end)].copy()

final_forecast['Frequency'] = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Total_Claim'] = final_forecast['Daily_Total']

# Safe Severity Calculation
final_forecast['Severity'] = np.where(final_forecast['Frequency'] > 0, 
                                      final_forecast['Total_Claim'] / final_forecast['Frequency'], 
                                      0)

# ==========================================
# 6. EXPORT KAGGLE CSV
# ==========================================
print("6. Formatting and Exporting Kaggle Submission...")

formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_golden_ensemble_ema.csv', index=False)

print("\n--- FINAL FORECAST (75/25 Golden Ensemble Architecture) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_golden_ensemble_ema.csv' saved successfully!")

1. Processing Raw Data to Weekly Level...
2. Generating Features for DLinear (EMA) and LightGBM (Raw Lags)...
3. Training Ensembled Models...
4. Executing Recursive Forecast & Golden Blend (75% DL / 25% LGBM)...
5. Apportioning to Daily and Rolling up to Exact Calendar Months...
6. Formatting and Exporting Kaggle Submission...

--- FINAL FORECAST (75/25 Golden Ensemble Architecture) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        206  5.633416e+07  1.160484e+10
1      2025-09        230  5.298020e+07  1.218545e+10
2      2025-10        235  5.333180e+07  1.253297e+10
3      2025-11        228  5.349327e+07  1.219646e+10
4      2025-12        237  5.306557e+07  1.257654e+10

File 'submission_golden_ensemble_ema.csv' saved successfully!


In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, ElasticNet, QuantileRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
WINDOW_SIZE       = 4        # STL seasonal period (weeks)
N_LAGS            = 12       # Lag features (covers ~1 quarter)
N_CV_SPLITS       = 5        # TimeSeriesSplit folds
CV_GAP            = 4        # Weeks gap between train/val (prevent leakage)
RIDGE_ALPHA       = 1.0      # Regularization strength
FORECAST_START    = '2025-08'
FORECAST_END      = '2025-12'
OUTPUT_CSV        = 'submission_dlinear_improved.csv'

# ============================================================
# 1. RAW DATA → WEEKLY AGGREGATION
# ============================================================
print("=" * 60)
print("STEP 1: Processing Raw Data to Weekly Level...")
print("=" * 60)

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()
weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# Log transform for numerical stability
weekly_df['Frequency_Log']   = np.log1p(weekly_df['Frequency'])
weekly_df['Total_Claim_Log'] = np.log1p(weekly_df['Total_Claim'])

# Model severity directly (log-space subtraction = ratio in real space)
# This avoids dividing noisy prediction by noisy prediction later
weekly_df['Severity_Log'] = weekly_df['Total_Claim_Log'] - weekly_df['Frequency_Log']

print(f"  Weekly observations: {len(weekly_df)}")
print(f"  Date range: {weekly_df['Week_End_Date'].min().date()} → {weekly_df['Week_End_Date'].max().date()}")

# ============================================================
# 2. DAY-OF-WEEK WEIGHTS (for smarter daily apportionment)
# ============================================================
print("\nSTEP 2: Computing Day-of-Week Claim Weights...")

dow_freq  = valid_claims.groupby(valid_claims.index.dayofweek)['Nomor Polis'].count()
dow_total = valid_claims.groupby(valid_claims.index.dayofweek)['Nominal Klaim Yang Disetujui'].sum()

# Reindex to ensure all 7 days present, fill missing with 0
dow_freq  = dow_freq.reindex(range(7), fill_value=0)
dow_total = dow_total.reindex(range(7), fill_value=0)

# Normalize (if all zero, fall back to uniform)
dow_freq_w  = dow_freq  / dow_freq.sum()  if dow_freq.sum()  > 0 else pd.Series([1/7]*7)
dow_total_w = dow_total / dow_total.sum() if dow_total.sum() > 0 else pd.Series([1/7]*7)

print(f"  DOW weights (Freq)  Mon–Sun: {[f'{w:.3f}' for w in dow_freq_w]}")

# ============================================================
# 3. STL DECOMPOSITION (replaces simple rolling mean)
# ============================================================
print("\nSTEP 3: STL Decomposition on Log-Scaled Data...")

TARGETS = ['Frequency_Log', 'Total_Claim_Log', 'Severity_Log']

def run_stl(series, period=WINDOW_SIZE):
    """Robust STL decomposition; returns trend, seasonal, residual."""
    stl = STL(series, period=period, robust=True)
    res = stl.fit()
    return res.trend, res.seasonal, res.resid

for col in TARGETS:
    trend, seasonal, resid = run_stl(weekly_df[col].values)
    weekly_df[f'{col}_Trend']    = trend
    weekly_df[f'{col}_Seasonal'] = seasonal
    weekly_df[f'{col}_Resid']    = resid

# ============================================================
# 4. FEATURE ENGINEERING
# ============================================================
print("\nSTEP 4: Building Rich Feature Set...")

def build_features(df, targets=TARGETS, n_lags=N_LAGS):
    """
    For each target, create:
      - Lags 1..N_LAGS on the raw log series
      - Lags 1..4 on Trend, Seasonal, Resid components
      - Rolling stats (std, min, max over 4 weeks)
      - Calendar features
    """
    out = df.copy()

    # --- Lagged raw log values ---
    for col in targets:
        for lag in range(1, n_lags + 1):
            out[f'{col}_Lag{lag}'] = out[col].shift(lag)

    # --- Lagged decomposition components ---
    for col in targets:
        for comp in ['Trend', 'Seasonal', 'Resid']:
            for lag in range(1, 5):
                out[f'{col}_{comp}_Lag{lag}'] = out[f'{col}_{comp}'].shift(lag)

    # --- Rolling statistics on log values ---
    for col in targets:
        out[f'{col}_Roll_Std'] = out[col].rolling(4).std()
        out[f'{col}_Roll_Min'] = out[col].rolling(4).min()
        out[f'{col}_Roll_Max'] = out[col].rolling(4).max()

    # --- Calendar features ---
    out['week_of_year'] = out['Week_End_Date'].dt.isocalendar().week.astype(int)
    out['month']        = out['Week_End_Date'].dt.month
    out['quarter']      = out['Week_End_Date'].dt.quarter
    out['is_q4']        = (out['month'] >= 10).astype(int)   # Q4 claims spike
    out['is_year_end']  = (out['month'] == 12).astype(int)
    out['sin_week']     = np.sin(2 * np.pi * out['week_of_year'] / 52)
    out['cos_week']     = np.cos(2 * np.pi * out['week_of_year'] / 52)
    out['sin_month']    = np.sin(2 * np.pi * out['month'] / 12)
    out['cos_month']    = np.cos(2 * np.pi * out['month'] / 12)

    return out

weekly_df = build_features(weekly_df)
train_df  = weekly_df.dropna().copy().reset_index(drop=True)
print(f"  Training rows after dropna: {len(train_df)}")

# ============================================================
# 5. FEATURE SETS PER TARGET
# ============================================================
CALENDAR_FEATS = ['week_of_year', 'month', 'quarter', 'is_q4',
                  'is_year_end', 'sin_week', 'cos_week', 'sin_month', 'cos_month']

def get_feature_cols(target, n_lags=N_LAGS):
    lag_cols   = [f'{target}_Lag{i}' for i in range(1, n_lags + 1)]
    decomp_cols = [f'{target}_{c}_Lag{i}'
                   for c in ['Trend', 'Seasonal', 'Resid']
                   for i in range(1, 5)]
    roll_cols  = [f'{target}_Roll_Std', f'{target}_Roll_Min', f'{target}_Roll_Max']
    return lag_cols + decomp_cols + roll_cols + CALENDAR_FEATS

FEATURE_SETS = {t: get_feature_cols(t) for t in TARGETS}

# ============================================================
# 6. TIME-SERIES CROSS-VALIDATION
# ============================================================
print("\nSTEP 5: Time-Series Cross-Validation...")

tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS, gap=CV_GAP)

cv_results = {}
for target in TARGETS:
    X = train_df[FEATURE_SETS[target]].values
    y = train_df[target].values
    maes = []
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X)):
        m = Ridge(alpha=RIDGE_ALPHA)
        m.fit(X[tr_idx], y[tr_idx])
        preds = m.predict(X[val_idx])
        # Evaluate in original space
        mae = mean_absolute_error(np.expm1(y[val_idx]), np.expm1(preds))
        maes.append(mae)
    cv_results[target] = np.mean(maes)
    print(f"  {target:25s} | CV MAE (original space): {cv_results[target]:,.2f}")

# ============================================================
# 7. DIRECT MULTI-HORIZON FORECASTING
#    Train one model per forecast horizon (no error compounding)
# ============================================================
print("\nSTEP 6: Direct Multi-Horizon Model Training...")

last_hist_date = weekly_df['Week_End_Date'].max()
target_end_dt  = pd.to_datetime('2025-12-31')
forecast_dates = pd.date_range(
    start=last_hist_date + pd.Timedelta(days=7),
    end=target_end_dt   + pd.Timedelta(days=7),
    freq='W-SUN'
)
H = len(forecast_dates)
print(f"  Forecast horizons: {H} weeks")

# horizon_models[target][h] = fitted Ridge model for horizon h (1-indexed)
horizon_models        = {t: {} for t in TARGETS}
horizon_models_lower  = {t: {} for t in TARGETS}
horizon_models_upper  = {t: {} for t in TARGETS}

X_full = {t: train_df[FEATURE_SETS[t]].values for t in TARGETS}
y_full = {t: train_df[t].values              for t in TARGETS}

for target in TARGETS:
    X = X_full[target]
    y = y_full[target]
    n = len(y)
    for h in range(1, H + 1):
        # Align: X[i] predicts y[i+h]
        X_h = X[:n - h]
        y_h = y[h:]
        if len(X_h) < 10:
            # Fall back to h=1 model if not enough data
            X_h = X[:n - 1]
            y_h = y[1:]

        # Point estimate
        m = Ridge(alpha=RIDGE_ALPHA)
        m.fit(X_h, y_h)
        horizon_models[target][h] = m

        # Prediction intervals (80% PI via quantile regression)
        try:
            ml = QuantileRegressor(quantile=0.1, alpha=0.1, solver='highs')
            mu = QuantileRegressor(quantile=0.9, alpha=0.1, solver='highs')
            ml.fit(X_h, y_h)
            mu.fit(X_h, y_h)
            horizon_models_lower[target][h] = ml
            horizon_models_upper[target][h] = mu
        except Exception:
            horizon_models_lower[target][h] = m
            horizon_models_upper[target][h] = m

    print(f"  ✓ {target}: {H} horizon models trained")

# ============================================================
# 8. GENERATE FORECASTS
# ============================================================
print("\nSTEP 7: Generating Direct Forecasts...")

# Get the last feature row from training data
last_feature_rows = {t: train_df[FEATURE_SETS[t]].values[-1:] for t in TARGETS}

weekly_predictions = []
for h_idx, week_end in enumerate(forecast_dates):
    h = h_idx + 1
    row = {}
    for target in TARGETS:
        X_row = last_feature_rows[target]
        pred_log   = horizon_models[target][h].predict(X_row)[0]
        pred_lower = horizon_models_lower[target][h].predict(X_row)[0]
        pred_upper = horizon_models_upper[target][h].predict(X_row)[0]

        row[f'{target}_log']   = pred_log
        row[f'{target}_lower'] = pred_lower
        row[f'{target}_upper'] = pred_upper

    # Convert back to original space
    freq_pred  = max(0, np.expm1(row['Frequency_Log_log']))
    sev_pred   = max(0, np.expm1(row['Severity_Log_log']))
    total_pred = max(0, np.expm1(row['Total_Claim_Log_log']))

    # Consistency check: use severity × frequency as alternative total
    total_from_sev = freq_pred * sev_pred
    # Blend: weighted average (model total vs derived total)
    total_final = 0.6 * total_pred + 0.4 * total_from_sev

    # Prediction intervals
    freq_lower  = max(0, np.expm1(row['Frequency_Log_lower']))
    freq_upper  = max(0, np.expm1(row['Frequency_Log_upper']))
    total_lower = max(0, np.expm1(row['Total_Claim_Log_lower']))
    total_upper = max(0, np.expm1(row['Total_Claim_Log_upper']))

    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date':   week_end,
        'Horizon_Weeks':   h,
        'Frequency':       freq_pred,
        'Frequency_Lower': freq_lower,
        'Frequency_Upper': freq_upper,
        'Severity':        sev_pred,
        'Total_Claim':     total_final,
        'Total_Lower':     total_lower,
        'Total_Upper':     total_upper,
    })

pred_df = pd.DataFrame(weekly_predictions)

# ============================================================
# 9. DAILY APPORTIONMENT WITH DOW WEIGHTS
# ============================================================
print("\nSTEP 8: Apportioning to Daily (DOW-weighted)...")

daily_records = []
for _, row in pred_df.iterrows():
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    for d in days:
        dow = d.dayofweek
        daily_records.append({
            'Date':        d,
            'Daily_Freq':  row['Frequency']   * dow_freq_w[dow],
            'Daily_Total': row['Total_Claim']  * dow_total_w[dow],
            'Daily_Freq_Lower':  row['Frequency_Lower'] * dow_freq_w[dow],
            'Daily_Freq_Upper':  row['Frequency_Upper'] * dow_freq_w[dow],
            'Daily_Total_Lower': row['Total_Lower']     * dow_total_w[dow],
            'Daily_Total_Upper': row['Total_Upper']     * dow_total_w[dow],
        })

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

# ============================================================
# 10. MONTHLY ROLL-UP
# ============================================================
print("\nSTEP 9: Rolling Up to Calendar Months...")

monthly_forecast = daily_df.groupby('Month_Period').agg(
    Daily_Freq        =('Daily_Freq',        'sum'),
    Daily_Total       =('Daily_Total',       'sum'),
    Daily_Freq_Lower  =('Daily_Freq_Lower',  'sum'),
    Daily_Freq_Upper  =('Daily_Freq_Upper',  'sum'),
    Daily_Total_Lower =('Daily_Total_Lower', 'sum'),
    Daily_Total_Upper =('Daily_Total_Upper', 'sum'),
).reset_index()

target_start = pd.Period(FORECAST_START, freq='M')
target_end   = pd.Period(FORECAST_END,   freq='M')

final_forecast = monthly_forecast[
    (monthly_forecast['Month_Period'] >= target_start) &
    (monthly_forecast['Month_Period'] <= target_end)
].copy()

final_forecast['Frequency']       = np.round(final_forecast['Daily_Freq']).astype(int)
final_forecast['Frequency_Lower'] = np.round(final_forecast['Daily_Freq_Lower']).astype(int)
final_forecast['Frequency_Upper'] = np.round(final_forecast['Daily_Freq_Upper']).astype(int)
final_forecast['Total_Claim']     = final_forecast['Daily_Total']
final_forecast['Total_Lower']     = final_forecast['Daily_Total_Lower']
final_forecast['Total_Upper']     = final_forecast['Daily_Total_Upper']
final_forecast['Severity']        = final_forecast['Total_Claim'] / final_forecast['Frequency']
final_forecast['Severity_Lower']  = final_forecast['Total_Lower']  / final_forecast['Frequency_Upper']
final_forecast['Severity_Upper']  = final_forecast['Total_Upper']  / final_forecast['Frequency_Lower']

# ============================================================
# 11. EXPORT — SUBMISSION + FULL DETAIL
# ============================================================
print("\nSTEP 10: Exporting Results...")

# --- Submission format (same as original) ---
formatted_data = []
for _, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f'{m_id}_Claim_Frequency', 'value': row['Frequency']})
    formatted_data.append({'id': f'{m_id}_Claim_Severity',  'value': row['Severity']})
    formatted_data.append({'id': f'{m_id}_Total_Claim',     'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv(OUTPUT_CSV, index=False)

# --- Full detailed output with prediction intervals ---
detail_cols = [
    'Month_Period',
    'Frequency', 'Frequency_Lower', 'Frequency_Upper',
    'Severity',  'Severity_Lower',  'Severity_Upper',
    'Total_Claim','Total_Lower',    'Total_Upper'
]
final_forecast[detail_cols].to_csv('submission_dlinear_detailed.csv', index=False)

# ============================================================
# 12. SUMMARY REPORT
# ============================================================
print("\n" + "=" * 60)
print("FINAL FORECAST — Improved DLinear (Direct Multi-Horizon + STL)")
print("=" * 60)

for _, row in final_forecast.iterrows():
    print(f"\n  {row['Month_Period']}")
    print(f"    Frequency : {row['Frequency']:>8,d}  [{row['Frequency_Lower']:,d} – {row['Frequency_Upper']:,d}]")
    print(f"    Severity  : {row['Severity']:>15,.0f}  [{row['Severity_Lower']:,.0f} – {row['Severity_Upper']:,.0f}]")
    print(f"    Total     : {row['Total_Claim']:>15,.0f}  [{row['Total_Lower']:,.0f} – {row['Total_Upper']:,.0f}]")

print("\n" + "=" * 60)
print("CV Summary (MAE in original scale):")
for target, mae in cv_results.items():
    print(f"  {target:25s}: {mae:,.2f}")
print("=" * 60)
print(f"\n✓ Submission saved  → {OUTPUT_CSV}")
print(f"✓ Detailed saved    → submission_dlinear_detailed.csv")
print("\nKey upgrades applied:")
print("  [1] STL decomposition (robust, seasonal-aware) replaces rolling mean")
print("  [2] Direct multi-horizon training — zero error compounding")
print("  [3] 12-lag + calendar + rolling-stat features (was: 4-lag only)")
print("  [4] Time-series cross-validation (5-fold, gap=4 weeks)")
print("  [5] 80% prediction intervals via Quantile Regression")
print("  [6] Severity modeled directly in log space (no noisy division)")
print("  [7] DOW-weighted daily apportionment (replaces uniform ÷7)")

STEP 1: Processing Raw Data to Weekly Level...
  Weekly observations: 83
  Date range: 2024-01-07 → 2025-08-03

STEP 2: Computing Day-of-Week Claim Weights...
  DOW weights (Freq)  Mon–Sun: ['0.194', '0.203', '0.165', '0.166', '0.145', '0.091', '0.035']

STEP 3: STL Decomposition on Log-Scaled Data...

STEP 4: Building Rich Feature Set...
  Training rows after dropna: 71

STEP 5: Time-Series Cross-Validation...
  Frequency_Log             | CV MAE (original space): 11.24
  Total_Claim_Log           | CV MAE (original space): 843,406,956.80
  Severity_Log              | CV MAE (original space): 16,220,414.73

STEP 6: Direct Multi-Horizon Model Training...
  Forecast horizons: 22 weeks
  ✓ Frequency_Log: 22 horizon models trained
  ✓ Total_Claim_Log: 22 horizon models trained
  ✓ Severity_Log: 22 horizon models trained

STEP 7: Generating Direct Forecasts...

STEP 8: Apportioning to Daily (DOW-weighted)...

STEP 9: Rolling Up to Calendar Months...

STEP 10: Exporting Results...

FINAL FO

In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from statsmodels.tsa.seasonal import STL
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# PHILOSOPHY: Small dataset → simple model wins.
# Keep: log transform, recursive forecasting (works well here)
# Fix:  STL decomposition, smarter lags, severity modeled directly
# Drop: 12 lags, quantile regression, calendar features (overfit risk)
# ============================================================

WINDOW_SIZE    = 4      # STL period (4 weeks ≈ monthly seasonality)
N_LAGS         = 4      # Keep original lag count — data is too small for more
RIDGE_ALPHA    = 10.0   # Stronger regularization to fight overfitting
FORECAST_START = '2025-08'
FORECAST_END   = '2025-12'
OUTPUT_CSV     = 'submission_dlinear_v3.csv'

# ============================================================
# 1. RAW DATA → WEEKLY AGGREGATION
# ============================================================
print("STEP 1: Weekly Aggregation...")

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()
weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# Log transform
weekly_df['Frequency_Log']   = np.log1p(weekly_df['Frequency'])
weekly_df['Total_Claim_Log'] = np.log1p(weekly_df['Total_Claim'])

# *** FIX: Model severity directly in log space ***
# log(severity) = log(total) - log(freq)  →  avoids dividing two noisy preds
weekly_df['Severity_Log'] = weekly_df['Total_Claim_Log'] - weekly_df['Frequency_Log']

print(f"  Rows: {len(weekly_df)} | Range: {weekly_df['Week_End_Date'].min().date()} → {weekly_df['Week_End_Date'].max().date()}")

# ============================================================
# 2. DAY-OF-WEEK WEIGHTS
# ============================================================
print("STEP 2: Day-of-Week Weights...")

dow_freq  = valid_claims.groupby(valid_claims.index.dayofweek)['Nomor Polis'].count().reindex(range(7), fill_value=1)
dow_total = valid_claims.groupby(valid_claims.index.dayofweek)['Nominal Klaim Yang Disetujui'].sum().reindex(range(7), fill_value=1)
dow_freq_w  = dow_freq  / dow_freq.sum()
dow_total_w = dow_total / dow_total.sum()

# ============================================================
# 3. STL DECOMPOSITION  (replaces naive rolling mean)
#    robust=True makes it resistant to claim spikes/outliers
# ============================================================
print("STEP 3: STL Decomposition...")

TARGETS = ['Frequency_Log', 'Total_Claim_Log', 'Severity_Log']

for col in TARGETS:
    stl = STL(weekly_df[col], period=WINDOW_SIZE, robust=True)
    res = stl.fit()
    weekly_df[f'{col}_Trend']  = res.trend
    weekly_df[f'{col}_Resid']  = res.resid
    # Note: we intentionally DROP seasonal component —
    # 4-week seasonality is very noisy on small insurance data

# ============================================================
# 4. BUILD LAGGED FEATURES  (same count as original: 4 lags)
#    Only trend + resid lags — lean feature set to avoid overfit
# ============================================================
print("STEP 4: Building Lag Features...")

for col in TARGETS:
    for lag in range(1, N_LAGS + 1):
        weekly_df[f'{col}_Trend_Lag{lag}'] = weekly_df[f'{col}_Trend'].shift(lag)
        weekly_df[f'{col}_Resid_Lag{lag}'] = weekly_df[f'{col}_Resid'].shift(lag)

train_df = weekly_df.dropna().copy().reset_index(drop=True)
print(f"  Training rows: {len(train_df)}")

# ============================================================
# 5. TRAIN RIDGE MODELS  (stronger alpha = less overfit)
# ============================================================
print("STEP 5: Training Models...")

def train_model(df, target):
    trend_feats = [f'{target}_Trend_Lag{i}' for i in range(1, N_LAGS + 1)]
    resid_feats = [f'{target}_Resid_Lag{i}' for i in range(1, N_LAGS + 1)]
    m_trend = Ridge(alpha=RIDGE_ALPHA).fit(df[trend_feats], df[f'{target}_Trend'])
    m_resid = Ridge(alpha=RIDGE_ALPHA).fit(df[resid_feats], df[f'{target}_Resid'])
    return m_trend, m_resid, trend_feats, resid_feats

models = {}
for col in TARGETS:
    m_t, m_r, f_t, f_r = train_model(train_df, col)
    models[col] = {'trend': m_t, 'resid': m_r, 'trend_feats': f_t, 'resid_feats': f_r}
    print(f"  ✓ {col}")

# ============================================================
# 6. RECURSIVE WEEKLY FORECASTING
#    (kept from original — correct for small datasets)
# ============================================================
print("STEP 6: Recursive Forecasting...")

last_hist_date = weekly_df['Week_End_Date'].max()
forecast_dates = pd.date_range(
    start=last_hist_date + pd.Timedelta(days=7),
    end=pd.to_datetime('2025-12-31') + pd.Timedelta(days=7),
    freq='W-SUN'
)

current_history = weekly_df.copy()
weekly_predictions = []

for week_end in forecast_dates:
    last4 = current_history.tail(4)
    preds_log = {}

    for col in TARGETS:
        m     = models[col]
        row_t = pd.DataFrame([{f: last4[f'{col}_Trend'].iloc[-int(f[-1])] for f in m['trend_feats']}])
        row_r = pd.DataFrame([{f: last4[f'{col}_Resid'].iloc[-int(f[-1])] for f in m['resid_feats']}])
        pred_log = m['trend'].predict(row_t)[0] + m['resid'].predict(row_r)[0]
        preds_log[col] = pred_log

    # Convert from log space
    freq_pred  = max(0, np.expm1(preds_log['Frequency_Log']))
    sev_pred   = max(0, np.expm1(preds_log['Severity_Log']))
    total_pred = max(0, np.expm1(preds_log['Total_Claim_Log']))

    # *** FIX: Blend model-total with freq×severity for consistency ***
    total_final = 0.5 * total_pred + 0.5 * (freq_pred * sev_pred)

    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date':   week_end,
        'Frequency':       freq_pred,
        'Severity':        sev_pred,
        'Total_Claim':     total_final,
    })

    # Update history for next recursive step
    new_row = {
        'Week_End_Date':     week_end,
        'Frequency':         freq_pred,
        'Total_Claim':       total_final,
        'Frequency_Log':     preds_log['Frequency_Log'],
        'Total_Claim_Log':   preds_log['Total_Claim_Log'],
        'Severity_Log':      preds_log['Severity_Log'],
    }
    hist_plus = pd.concat([current_history, pd.DataFrame([new_row])], ignore_index=True)

    # Re-run STL on growing history
    for col in TARGETS:
        stl = STL(hist_plus[col].ffill(), period=WINDOW_SIZE, robust=True)
        res = stl.fit()
        hist_plus[f'{col}_Trend'] = res.trend
        hist_plus[f'{col}_Resid'] = res.resid

    # Recompute lags on updated history
    for col in TARGETS:
        for lag in range(1, N_LAGS + 1):
            hist_plus[f'{col}_Trend_Lag{lag}'] = hist_plus[f'{col}_Trend'].shift(lag)
            hist_plus[f'{col}_Resid_Lag{lag}'] = hist_plus[f'{col}_Resid'].shift(lag)

    current_history = hist_plus.copy()

pred_df = pd.DataFrame(weekly_predictions)

# ============================================================
# 7. DAILY APPORTIONMENT (DOW-weighted)
# ============================================================
print("STEP 7: Daily Apportionment...")

daily_records = []
for _, row in pred_df.iterrows():
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    for d in days:
        dow = d.dayofweek
        daily_records.append({
            'Date':        d,
            'Daily_Freq':  row['Frequency']  * dow_freq_w[dow],
            'Daily_Total': row['Total_Claim'] * dow_total_w[dow],
        })

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

# ============================================================
# 8. MONTHLY ROLL-UP
# ============================================================
print("STEP 8: Monthly Roll-up...")

monthly = daily_df.groupby('Month_Period').agg(
    Daily_Freq=('Daily_Freq', 'sum'),
    Daily_Total=('Daily_Total', 'sum'),
).reset_index()

final = monthly[
    (monthly['Month_Period'] >= pd.Period(FORECAST_START, freq='M')) &
    (monthly['Month_Period'] <= pd.Period(FORECAST_END,   freq='M'))
].copy()

final['Frequency']  = np.round(final['Daily_Freq']).astype(int)
final['Total_Claim'] = final['Daily_Total']
final['Severity']   = final['Total_Claim'] / final['Frequency']

# ============================================================
# 9. EXPORT — submission format
# ============================================================
print("STEP 9: Exporting...")

rows = []
for _, row in final.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    rows.append({'id': f'{m_id}_Claim_Frequency', 'value': row['Frequency']})
    rows.append({'id': f'{m_id}_Claim_Severity',  'value': row['Severity']})
    rows.append({'id': f'{m_id}_Total_Claim',     'value': row['Total_Claim']})

submission_df = pd.DataFrame(rows)
submission_df.to_csv(OUTPUT_CSV, index=False)

# ============================================================
# 10. PRINT RESULTS
# ============================================================
print("\n" + "=" * 55)
print("FINAL FORECAST — DLinear v3 (STL + Stronger Ridge)")
print("=" * 55)
print(f"{'Month':<12} {'Frequency':>12} {'Severity':>18} {'Total Claim':>20}")
print("-" * 55)
for _, row in final.iterrows():
    print(f"{str(row['Month_Period']):<12} {row['Frequency']:>12,} {row['Severity']:>18,.0f} {row['Total_Claim']:>20,.0f}")
print("=" * 55)
print(f"\n✓ Saved → {OUTPUT_CSV}")
print("\nChanges vs v2 (overfit):")
print("  [kept]    Log transform + recursive forecasting")
print("  [upgrade] STL replaces rolling mean (robust to spikes)")
print("  [upgrade] Ridge alpha 1.0 → 10.0 (stronger regularization)")
print("  [upgrade] Severity modeled in log space directly")
print("  [upgrade] DOW-weighted daily split")
print("  [removed] 12 lags, calendar feats, quantile reg (all overfit on small data)")

STEP 1: Weekly Aggregation...
  Rows: 83 | Range: 2024-01-07 → 2025-08-03
STEP 2: Day-of-Week Weights...
STEP 3: STL Decomposition...
STEP 4: Building Lag Features...
  Training rows: 79
STEP 5: Training Models...
  ✓ Frequency_Log
  ✓ Total_Claim_Log
  ✓ Severity_Log
STEP 6: Recursive Forecasting...
STEP 7: Daily Apportionment...
STEP 8: Monthly Roll-up...
STEP 9: Exporting...

FINAL FORECAST — DLinear v3 (STL + Stronger Ridge)
Month           Frequency           Severity          Total Claim
-------------------------------------------------------
2025-08               213         47,835,226       10,188,903,034
2025-09               235         49,600,952       11,656,223,678
2025-10               240         50,339,572       12,081,497,375
2025-11               221         50,112,008       11,074,753,812
2025-12               244         50,224,305       12,254,730,449

✓ Saved → submission_dlinear_v3.csv

Changes vs v2 (overfit):
  [kept]    Log transform + recursive forecasting
  

In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# STRATEGY FOR ~70 WEEKLY ROWS
# ============================================================
# Rule: With <100 obs, classical statistical models beat ML.
# Approach: Weighted ensemble of 4 methods:
#   [1] DLinear LogScaled   — your best baseline (5.9%)
#   [2] Holt-Winters ETS    — captures trend + seasonality
#   [3] Theta Method        — M3 competition winner, robust
#   [4] SARIMA              — best-fit ARIMA with seasonal term
#
# Weights chosen by time-series CV on last 8 weeks of history.
# Final prediction = weighted average in log space.
# ============================================================

FORECAST_START = '2025-08'
FORECAST_END   = '2025-12'
OUTPUT_CSV     = 'submission_ensemble_best.csv'
SEED           = 42
np.random.seed(SEED)

# ============================================================
# 1. DATA PREPARATION
# ============================================================
print("=" * 60)
print("STEP 1: Data Preparation")
print("=" * 60)

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()
weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# Smooth extreme outliers (cap at 3-sigma in log space) before modeling
for col in ['Frequency', 'Total_Claim']:
    log_vals = np.log1p(weekly_df[col])
    mu, sigma = log_vals.mean(), log_vals.std()
    cap = np.expm1(mu + 3 * sigma)
    n_capped = (weekly_df[col] > cap).sum()
    weekly_df[col] = weekly_df[col].clip(upper=cap)
    if n_capped > 0:
        print(f"  Capped {n_capped} outlier weeks in {col} (3-sigma rule)")

weekly_df['Freq_Log']  = np.log1p(weekly_df['Frequency'])
weekly_df['Total_Log'] = np.log1p(weekly_df['Total_Claim'])

N = len(weekly_df)
print(f"  Weekly rows: {N}")
print(f"  Range: {weekly_df['Week_End_Date'].min().date()} → {weekly_df['Week_End_Date'].max().date()}")

# DOW weights for daily apportionment
dow_freq  = valid_claims.groupby(valid_claims.index.dayofweek)['Nomor Polis'].count().reindex(range(7), fill_value=1)
dow_total = valid_claims.groupby(valid_claims.index.dayofweek)['Nominal Klaim Yang Disetujui'].sum().reindex(range(7), fill_value=1)
dow_freq_w  = (dow_freq  / dow_freq.sum()).values
dow_total_w = (dow_total / dow_total.sum()).values

# ============================================================
# 2. DEFINE FORECAST HORIZON
# ============================================================
last_date      = weekly_df['Week_End_Date'].max()
forecast_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=7),
    end=pd.to_datetime('2025-12-31') + pd.Timedelta(days=7),
    freq='W-SUN'
)
H = len(forecast_dates)
print(f"  Forecast horizon: {H} weeks")

# ============================================================
# 3. MODEL DEFINITIONS
# ============================================================

def forecast_dlinear(series_log, H, window=4, alpha=1.0, n_lags=4):
    """
    Original DLinear LogScaled — your 5.9% baseline.
    Rolling mean decomposition + Ridge on trend/remainder.
    Recursive multi-step forecasting.
    """
    df = pd.DataFrame({'y': series_log})
    df['trend']  = df['y'].rolling(window=window, min_periods=1).mean()
    df['resid']  = df['y'] - df['trend']

    for lag in range(1, n_lags + 1):
        df[f't{lag}'] = df['trend'].shift(lag)
        df[f'r{lag}'] = df['resid'].shift(lag)
    df = df.dropna()

    t_feats = [f't{i}' for i in range(1, n_lags + 1)]
    r_feats = [f'r{i}' for i in range(1, n_lags + 1)]
    m_t = Ridge(alpha=alpha).fit(df[t_feats], df['trend'])
    m_r = Ridge(alpha=alpha).fit(df[r_feats], df['resid'])

    hist = list(series_log)
    preds = []
    for _ in range(H):
        s = pd.Series(hist)
        trend_hist = s.rolling(window=window, min_periods=1).mean().values
        resid_hist = (s - pd.Series(trend_hist)).values

        row_t = np.array([[trend_hist[-i] for i in range(1, n_lags + 1)]])
        row_r = np.array([[resid_hist[-i] for i in range(1, n_lags + 1)]])
        pred = m_t.predict(row_t)[0] + m_r.predict(row_r)[0]
        preds.append(pred)
        hist.append(pred)
    return np.array(preds)


def forecast_ets(series_log, H):
    """
    Holt-Winters Exponential Smoothing.
    Auto-selects additive vs multiplicative trend.
    Best for data with clear trend + seasonal patterns.
    """
    best_aic = np.inf
    best_pred = None
    for trend in ['add', None]:
        for seasonal in ['add', None]:
            try:
                sp = 4 if seasonal else None
                m = ExponentialSmoothing(
                    series_log,
                    trend=trend,
                    seasonal=seasonal,
                    seasonal_periods=sp,
                    initialization_method='estimated'
                ).fit(optimized=True, remove_bias=True)
                if m.aic < best_aic:
                    best_aic = m.aic
                    best_pred = m.forecast(H)
            except Exception:
                continue
    if best_pred is None:
        # Fallback: simple exponential smoothing
        m = ExponentialSmoothing(series_log, trend=None, seasonal=None).fit()
        best_pred = m.forecast(H)
    return np.array(best_pred)


def forecast_theta(series_log, H):
    """
    Theta Method — winner of M3 competition, excellent for small samples.
    Decomposes into two 'theta lines':
      Theta=0: removes curvature → captures long-term trend (linear regression)
      Theta=2: amplifies curvature → captures short-term dynamics (SES)
    Final forecast = mean of both lines.
    """
    y = np.array(series_log)
    n = len(y)
    t = np.arange(1, n + 1)

    # Theta line 0 = OLS linear trend (deseasonalized long-run)
    p    = np.polyfit(t, y, 1)
    t_fut = np.arange(n + 1, n + H + 1)
    theta0_forecast = np.polyval(p, t_fut)

    # Theta line 2 = Simple Exponential Smoothing with optimal alpha
    best_alpha, best_sse = 0.1, np.inf
    for a in np.linspace(0.01, 0.99, 50):
        level = y[0]
        sse = 0.0
        for val in y[1:]:
            err   = val - level
            sse  += err ** 2
            level = a * val + (1 - a) * level
        if sse < best_sse:
            best_sse, best_alpha = sse, a

    # SES forecast: the SES forecast from last level is constant
    level = y[0]
    for val in y[1:]:
        level = best_alpha * val + (1 - best_alpha) * level
    # Drift adjustment: add half the linear slope per step
    slope = p[0]
    theta2_forecast = np.array([level + slope * 0.5 * i for i in range(1, H + 1)])

    return 0.5 * theta0_forecast + 0.5 * theta2_forecast


def forecast_sarima(series_log, H):
    """
    Auto-SARIMA: tries a small grid of (p,d,q)(P,D,Q,4) models,
    picks lowest AIC. Seasonal period=4 (monthly within quarters).
    Constrained grid to avoid overfitting on 70 rows.
    """
    best_aic  = np.inf
    best_pred = None

    # Constrained grid: max 5 params total to avoid overfit
    configs = [
        (1,1,1, 0,0,0),
        (1,1,0, 0,0,0),
        (0,1,1, 0,0,0),
        (1,1,1, 1,0,0),
        (1,1,1, 0,1,0),
        (1,1,1, 1,1,0),
        (0,1,1, 1,1,0),
        (2,1,0, 0,0,0),
        (0,1,2, 0,0,0),
    ]
    for (p,d,q,P,D,Q) in configs:
        try:
            m = SARIMAX(
                series_log,
                order=(p, d, q),
                seasonal_order=(P, D, Q, 4),
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(disp=False)
            if m.aic < best_aic:
                best_aic  = m.aic
                best_pred = m.forecast(H)
        except Exception:
            continue

    if best_pred is None:
        # Fallback: random walk
        best_pred = np.full(H, series_log[-1])
    return np.array(best_pred)


# ============================================================
# 4. FIND OPTIMAL ENSEMBLE WEIGHTS VIA TIME-SERIES CV
#    Use last 8 weeks as validation, rest as training.
#    Minimize MAE in original (non-log) space.
# ============================================================
print("\nSTEP 2: Optimizing Ensemble Weights via Walk-Forward CV...")

VAL_WEEKS = 8   # Hold out last 8 weeks for weight optimization

def evaluate_weights(weights, preds_matrix, y_true_log):
    """Weighted blend → MAE in original space."""
    blend_log = np.average(preds_matrix, axis=0, weights=weights)
    return np.mean(np.abs(np.expm1(y_true_log) - np.expm1(blend_log)))

target_logs = {
    'Frequency_Log':   weekly_df['Freq_Log'].values,
    'Total_Claim_Log': weekly_df['Total_Log'].values,
}

final_weights = {}
METHOD_NAMES = ['DLinear', 'ETS', 'Theta', 'SARIMA']

for tname, series in target_logs.items():
    print(f"\n  [{tname}]")
    train_s = series[:-VAL_WEEKS]
    val_s   = series[-VAL_WEEKS:]

    method_preds = []
    for name, fn in zip(METHOD_NAMES, [
        lambda s: forecast_dlinear(s, VAL_WEEKS),
        lambda s: forecast_ets(s, VAL_WEEKS),
        lambda s: forecast_theta(s, VAL_WEEKS),
        lambda s: forecast_sarima(s, VAL_WEEKS),
    ]):
        try:
            p = fn(train_s)
            p = np.clip(p, series.min() - 1, series.max() + 1)
        except Exception as e:
            print(f"    {name} failed: {e} — using naive")
            p = np.full(VAL_WEEKS, train_s[-1])
        mae = np.mean(np.abs(np.expm1(val_s) - np.expm1(p)))
        print(f"    {name:10s}: MAE = {mae:,.2f}")
        method_preds.append(p)

    preds_matrix = np.vstack(method_preds)  # shape (4, VAL_WEEKS)

    # Greedy weight search: grid over simplex
    # For 4 models → try all combos of [0.0, 0.1, ..., 1.0] summing to 1
    best_mae = np.inf
    best_w   = np.array([0.25, 0.25, 0.25, 0.25])

    for w0 in np.arange(0, 1.01, 0.1):
        for w1 in np.arange(0, 1.01 - w0, 0.1):
            for w2 in np.arange(0, 1.01 - w0 - w1, 0.1):
                w3 = 1.0 - w0 - w1 - w2
                if w3 < -0.001:
                    continue
                w3 = max(0, w3)
                w  = np.array([w0, w1, w2, w3])
                if w.sum() < 0.99:
                    continue
                mae = evaluate_weights(w, preds_matrix, val_s)
                if mae < best_mae:
                    best_mae = mae
                    best_w   = w.copy()

    final_weights[tname] = best_w
    print(f"    Best weights → DLinear:{best_w[0]:.1f} ETS:{best_w[1]:.1f} Theta:{best_w[2]:.1f} SARIMA:{best_w[3]:.1f}")
    print(f"    Ensemble MAE = {best_mae:,.2f}")

# ============================================================
# 5. FULL-DATA FORECAST WITH OPTIMAL WEIGHTS
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: Generating Final Forecasts on Full History...")
print("=" * 60)

weekly_predictions = []
full_preds = {}

for tname, series in target_logs.items():
    weights = final_weights[tname]
    method_preds = []
    for name, fn in zip(METHOD_NAMES, [
        lambda s: forecast_dlinear(s, H),
        lambda s: forecast_ets(s, H),
        lambda s: forecast_theta(s, H),
        lambda s: forecast_sarima(s, H),
    ]):
        try:
            p = fn(series)
        except Exception:
            p = np.full(H, series[-1])
        method_preds.append(p)

    blend = np.average(np.vstack(method_preds), axis=0, weights=weights)
    full_preds[tname] = np.maximum(0, np.expm1(blend))
    print(f"  ✓ {tname}")

# ============================================================
# 6. BUILD WEEKLY PREDICTION TABLE
#    Blend model-total with freq×severity for internal consistency
# ============================================================
freq_arr  = full_preds['Frequency_Log']
total_arr = full_preds['Total_Claim_Log']
sev_arr   = np.where(freq_arr > 0, total_arr / freq_arr, 0)

for i, week_end in enumerate(forecast_dates):
    freq  = max(0, freq_arr[i])
    total = max(0, total_arr[i])
    sev   = max(0, sev_arr[i])
    # Blend direct total prediction with freq×sev for consistency
    total_final = 0.5 * total + 0.5 * (freq * sev)
    weekly_predictions.append({
        'Week_Start_Date': week_end - pd.Timedelta(days=6),
        'Week_End_Date':   week_end,
        'Frequency':       freq,
        'Severity':        sev,
        'Total_Claim':     total_final,
    })

pred_df = pd.DataFrame(weekly_predictions)

# ============================================================
# 7. DAILY APPORTIONMENT + MONTHLY ROLL-UP
# ============================================================
print("\nSTEP 4: Apportioning to Daily + Monthly Roll-up...")

daily_records = []
for _, row in pred_df.iterrows():
    days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
    for d in days:
        dow = d.dayofweek
        daily_records.append({
            'Date':        d,
            'Daily_Freq':  row['Frequency']  * dow_freq_w[dow],
            'Daily_Total': row['Total_Claim'] * dow_total_w[dow],
        })

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

monthly = daily_df.groupby('Month_Period').agg(
    Freq_Sum=('Daily_Freq', 'sum'),
    Total_Sum=('Daily_Total', 'sum'),
).reset_index()

final = monthly[
    (monthly['Month_Period'] >= pd.Period(FORECAST_START, freq='M')) &
    (monthly['Month_Period'] <= pd.Period(FORECAST_END, freq='M'))
].copy()

final['Frequency']   = np.round(final['Freq_Sum']).astype(int)
final['Total_Claim'] = final['Total_Sum']
final['Severity']    = final['Total_Claim'] / final['Frequency']

# ============================================================
# 8. EXPORT — EXACT SUBMISSION FORMAT
# ============================================================
rows = []
for _, row in final.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    rows.append({'id': f'{m_id}_Claim_Frequency', 'value': row['Frequency']})
    rows.append({'id': f'{m_id}_Claim_Severity',  'value': row['Severity']})
    rows.append({'id': f'{m_id}_Total_Claim',     'value': row['Total_Claim']})

submission_df = pd.DataFrame(rows)
submission_df.to_csv(OUTPUT_CSV, index=False)

# ============================================================
# 9. RESULTS SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("FINAL FORECAST — Optimal Weighted Ensemble")
print("=" * 60)
print(f"{'Month':<12} {'Frequency':>12} {'Severity':>18} {'Total Claim':>20}")
print("-" * 60)
for _, row in final.iterrows():
    print(f"{str(row['Month_Period']):<12} {row['Frequency']:>12,} "
          f"{row['Severity']:>18,.0f} {row['Total_Claim']:>20,.0f}")
print("=" * 60)

print("\nEnsemble Weights Summary:")
print(f"  {'Target':<22} {'DLinear':>9} {'ETS':>9} {'Theta':>9} {'SARIMA':>9}")
print("-" * 60)
for tname, w in final_weights.items():
    short = tname.replace('_Log', '')
    print(f"  {short:<22} {w[0]:>9.1f} {w[1]:>9.1f} {w[2]:>9.1f} {w[3]:>9.1f}")

print(f"\n✓ Saved → {OUTPUT_CSV}")
print("\nWhy this works for 70 rows:")
print("  • DLinear: your proven baseline, keeps error-compounding low")
print("  • ETS:     captures trend+seasonality with only 3-4 params")
print("  • Theta:   M3 competition winner — built for small samples")
print("  • SARIMA:  constrained grid prevents overfit, catches seasonality")
print("  • Weights: data-driven via 8-week walk-forward CV per target")
print("  • Outlier capping prevents single spike weeks from dominating")

STEP 1: Data Preparation
  Weekly rows: 83
  Range: 2024-01-07 → 2025-08-03
  Forecast horizon: 22 weeks

STEP 2: Optimizing Ensemble Weights via Walk-Forward CV...

  [Frequency_Log]
    DLinear   : MAE = 6.48
    ETS       : MAE = 6.37
    Theta     : MAE = 7.27
    SARIMA    : MAE = 6.43
    Best weights → DLinear:0.0 ETS:1.0 Theta:0.0 SARIMA:0.0
    Ensemble MAE = 6.37

  [Total_Claim_Log]
    DLinear   : MAE = 793,255,079.33
    ETS       : MAE = 797,162,748.93
    Theta     : MAE = 838,724,401.80
    SARIMA    : MAE = 801,358,696.52
    Best weights → DLinear:1.0 ETS:0.0 Theta:0.0 SARIMA:0.0
    Ensemble MAE = 793,255,079.33

STEP 3: Generating Final Forecasts on Full History...
  ✓ Frequency_Log
  ✓ Total_Claim_Log

STEP 4: Apportioning to Daily + Monthly Roll-up...

FINAL FORECAST — Optimal Weighted Ensemble
Month           Frequency           Severity          Total Claim
------------------------------------------------------------
2025-08               217         48,784,699 

In [11]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# IMPROVEMENT OVER v4 (5.49%)
# ============================================================
# [1] Optimize weights for MAPE, not MAE — matches evaluation metric
# [2] Log-normal bias correction: E[e^X] = e^(μ + σ²/2), not e^μ
# [3] scipy differential_evolution for true global weight optimum
# [4] 2 new models: Naive Seasonal + Drift — add diversity cheaply
# [5] Per-output weights: Frequency, Severity, Total optimized separately
# [6] Smarter validation: expand-window CV (not just last 8 weeks)
# [7] Winsorize outliers per-target, not uniform 3-sigma
# ============================================================

FORECAST_START = '2025-08'
FORECAST_END   = '2025-12'
OUTPUT_CSV     = 'submission_v5_mape_optimized.csv'
VAL_WEEKS      = 10   # slightly more CV data
SEASONAL_PERIOD = 4
np.random.seed(42)

# ============================================================
# 1. DATA PREP
# ============================================================
print("=" * 62)
print("STEP 1: Data Preparation")
print("=" * 62)

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()
weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)
weekly_df['Severity'] = weekly_df['Total_Claim'] / weekly_df['Frequency'].replace(0, np.nan)

N = len(weekly_df)
print(f"  Weekly rows: {N}")
print(f"  Range: {weekly_df['Week_End_Date'].min().date()} → {weekly_df['Week_End_Date'].max().date()}")

# Per-target winsorization (more adaptive than uniform 3-sigma)
def winsorize_series(s, lower_q=0.02, upper_q=0.98):
    lo, hi = np.nanquantile(s, lower_q), np.nanquantile(s, upper_q)
    n_clipped = ((s < lo) | (s > hi)).sum()
    if n_clipped > 0:
        print(f"    Winsorized {n_clipped} values → [{lo:.0f}, {hi:.0f}]")
    return np.clip(s, lo, hi)

for col in ['Frequency', 'Total_Claim', 'Severity']:
    print(f"  Winsorizing {col}...")
    weekly_df[col] = winsorize_series(weekly_df[col].values)

# Log transforms
weekly_df['Freq_Log']  = np.log1p(weekly_df['Frequency'])
weekly_df['Total_Log'] = np.log1p(weekly_df['Total_Claim'])
weekly_df['Sev_Log']   = np.log1p(weekly_df['Severity'])

# DOW weights
dow_freq  = valid_claims.groupby(valid_claims.index.dayofweek)['Nomor Polis'].count().reindex(range(7), fill_value=1)
dow_total = valid_claims.groupby(valid_claims.index.dayofweek)['Nominal Klaim Yang Disetujui'].sum().reindex(range(7), fill_value=1)
dow_freq_w  = (dow_freq  / dow_freq.sum()).values
dow_total_w = (dow_total / dow_total.sum()).values

# ============================================================
# 2. FORECAST HORIZON
# ============================================================
last_date      = weekly_df['Week_End_Date'].max()
forecast_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=7),
    end=pd.to_datetime('2025-12-31') + pd.Timedelta(days=7),
    freq='W-SUN'
)
H = len(forecast_dates)
print(f"  Forecast horizon: {H} weeks")

# ============================================================
# 3. MODEL LIBRARY (6 models for diversity)
# ============================================================

def forecast_dlinear(series_log, H, window=4, alpha=1.0, n_lags=4):
    """DLinear — your proven baseline."""
    df = pd.DataFrame({'y': series_log})
    df['trend'] = df['y'].rolling(window=window, min_periods=1).mean()
    df['resid'] = df['y'] - df['trend']
    for lag in range(1, n_lags + 1):
        df[f't{lag}'] = df['trend'].shift(lag)
        df[f'r{lag}'] = df['resid'].shift(lag)
    df = df.dropna()
    t_f = [f't{i}' for i in range(1, n_lags + 1)]
    r_f = [f'r{i}' for i in range(1, n_lags + 1)]
    m_t = Ridge(alpha=alpha).fit(df[t_f], df['trend'])
    m_r = Ridge(alpha=alpha).fit(df[r_f], df['resid'])
    hist = list(series_log)
    preds = []
    for _ in range(H):
        s = pd.Series(hist)
        tr = s.rolling(window=window, min_periods=1).mean().values
        re = (s - pd.Series(tr)).values
        pred = m_t.predict([[tr[-i] for i in range(1, n_lags+1)]])[0] \
             + m_r.predict([[re[-i] for i in range(1, n_lags+1)]])[0]
        preds.append(pred)
        hist.append(pred)
    return np.array(preds)


def forecast_ets(series_log, H):
    """Holt-Winters — auto-selects best config by AIC."""
    best_aic, best_pred = np.inf, None
    for trend in ['add', None]:
        for seasonal in ['add', None]:
            try:
                sp = SEASONAL_PERIOD if seasonal else None
                m = ExponentialSmoothing(
                    series_log, trend=trend, seasonal=seasonal,
                    seasonal_periods=sp, initialization_method='estimated'
                ).fit(optimized=True, remove_bias=True)
                if m.aic < best_aic:
                    best_aic, best_pred = m.aic, m.forecast(H)
            except Exception:
                continue
    return np.array(best_pred) if best_pred is not None else np.full(H, series_log[-1])


def forecast_theta(series_log, H):
    """Theta Method — M3 competition winner, built for small samples."""
    y  = np.array(series_log)
    n  = len(y)
    t  = np.arange(1, n + 1)
    p  = np.polyfit(t, y, 1)
    t_fut = np.arange(n + 1, n + H + 1)
    theta0 = np.polyval(p, t_fut)
    best_a, best_sse = 0.1, np.inf
    for a in np.linspace(0.01, 0.99, 99):
        level, sse = y[0], 0.0
        for val in y[1:]:
            sse += (val - level) ** 2
            level = a * val + (1 - a) * level
        if sse < best_sse:
            best_sse, best_a = sse, a
    level = y[0]
    for val in y[1:]:
        level = best_a * val + (1 - best_a) * level
    theta2 = np.array([level + p[0] * 0.5 * i for i in range(1, H + 1)])
    return 0.5 * theta0 + 0.5 * theta2


def forecast_sarima(series_log, H):
    """SARIMA — constrained grid to avoid overfit on 70 rows."""
    best_aic, best_pred = np.inf, None
    configs = [
        (1,1,1,0,0,0),(1,1,0,0,0,0),(0,1,1,0,0,0),
        (1,1,1,1,0,0),(1,1,1,0,1,0),(1,1,1,1,1,0),
        (0,1,1,1,1,0),(2,1,0,0,0,0),(0,1,2,0,0,0),
    ]
    for (p,d,q,P,D,Q) in configs:
        try:
            m = SARIMAX(series_log, order=(p,d,q),
                        seasonal_order=(P,D,Q,SEASONAL_PERIOD),
                        enforce_stationarity=False,
                        enforce_invertibility=False).fit(disp=False)
            if m.aic < best_aic:
                best_aic, best_pred = m.aic, m.forecast(H)
        except Exception:
            continue
    return np.array(best_pred) if best_pred is not None else np.full(H, series_log[-1])


def forecast_naive_seasonal(series_log, H, period=SEASONAL_PERIOD):
    """
    Seasonal Naive: repeats last observed seasonal cycle.
    Extremely simple but surprisingly strong baseline for short series.
    """
    s = np.array(series_log)
    preds = [s[-(period - (i % period))] for i in range(H)]
    return np.array(preds)


def forecast_drift(series_log, H):
    """
    Drift Model: extrapolates the average weekly change from full history.
    Captures long-run trend without any parameters to overfit.
    """
    s = np.array(series_log)
    slope = (s[-1] - s[0]) / (len(s) - 1)
    return np.array([s[-1] + slope * (i + 1) for i in range(H)])


MODEL_NAMES = ['DLinear', 'ETS', 'Theta', 'SARIMA', 'NaiveSeasonal', 'Drift']
MODEL_FNS   = [forecast_dlinear, forecast_ets, forecast_theta,
               forecast_sarima, forecast_naive_seasonal, forecast_drift]
M = len(MODEL_NAMES)

# ============================================================
# 4. MAPE-OPTIMIZED WEIGHTS via Differential Evolution
#    [KEY FIX vs v4] Evaluation metric is MAPE, not MAE.
#    scipy differential_evolution finds true global optimum.
#    Expand-window CV: trains on [0..split] for each split.
# ============================================================
print("\nSTEP 2: MAPE-Optimized Weight Search (Differential Evolution)...")

TARGET_CONFIGS = {
    'Frequency_Log': weekly_df['Freq_Log'].values,
    'Total_Claim_Log': weekly_df['Total_Log'].values,
    'Severity_Log': weekly_df['Sev_Log'].values,
}

def smape(y_true, y_pred):
    """Symmetric MAPE — more stable than regular MAPE when values near 0."""
    return np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8)))

def run_cv_preds(series, val_weeks, model_fns, model_names):
    """
    Expanding-window CV: trains on growing prefix, predicts next 1 step.
    Returns matrix of shape (M, val_weeks).
    """
    n = len(series)
    train_end = n - val_weeks
    all_preds = [[] for _ in model_fns]

    for step in range(val_weeks):
        train_s = series[:train_end + step]
        for j, fn in enumerate(model_fns):
            try:
                p = fn(train_s, 1)[0]
                p = np.clip(p, series.min() - 2, series.max() + 2)
            except Exception:
                p = train_s[-1]
            all_preds[j].append(p)

    return np.array(all_preds)  # shape (M, val_weeks)

final_weights = {}
cv_scores = {}

for tname, series in TARGET_CONFIGS.items():
    print(f"\n  [{tname}]")
    val_s   = series[-VAL_WEEKS:]
    val_orig = np.expm1(val_s)

    cv_preds = run_cv_preds(series, VAL_WEEKS, MODEL_FNS, MODEL_NAMES)

    # Individual model MAPE
    for j, name in enumerate(MODEL_NAMES):
        m_mape = mape(val_orig, np.expm1(cv_preds[j]))
        print(f"    {name:15s}: MAPE = {m_mape*100:.2f}%")

    # Global optimum via Differential Evolution on MAPE
    def objective(w):
        w = np.abs(w); w /= w.sum()
        blend_log  = np.dot(w, cv_preds)
        blend_orig = np.expm1(blend_log)
        return mape(val_orig, blend_orig)

    bounds = [(0, 1)] * M
    result = differential_evolution(
        objective, bounds,
        seed=42, maxiter=1000, tol=1e-8,
        popsize=20, mutation=(0.5, 1.5), recombination=0.9,
        polish=True
    )
    w_opt = np.abs(result.x); w_opt /= w_opt.sum()
    final_weights[tname] = w_opt

    blend_val  = np.expm1(np.dot(w_opt, cv_preds))
    best_mape  = mape(val_orig, blend_val) * 100
    cv_scores[tname] = best_mape

    print(f"    → Optimal weights: " + " ".join(f"{n}:{w:.2f}" for n,w in zip(MODEL_NAMES, w_opt)))
    print(f"    → Ensemble MAPE: {best_mape:.2f}%")

# ============================================================
# 5. LOG-NORMAL BIAS CORRECTION
#    [KEY FIX] log-transform introduces downward bias on back-transform.
#    True E[Y] = exp(μ + σ²/2), not exp(μ).
#    We estimate σ² from in-sample residuals.
# ============================================================
print("\nSTEP 3: Computing Log-Normal Bias Correction...")

bias_corrections = {}
for tname, series in TARGET_CONFIGS.items():
    # Fit best single model (weighted blend) on full data → get residuals
    full_preds_all = []
    for fn in MODEL_FNS:
        try:
            p = fn(series, 1)
        except Exception:
            p = [series[-1]]
        full_preds_all.append(p[0])
    w   = final_weights[tname]
    mu  = np.dot(w, full_preds_all)

    # Residual variance from last 20 in-sample 1-step predictions
    residuals = []
    for i in range(20, len(series)):
        train_s = series[:i]
        step_preds = []
        for fn in MODEL_FNS:
            try:
                p = fn(train_s, 1)[0]
            except Exception:
                p = train_s[-1]
            step_preds.append(p)
        blend = np.dot(w, step_preds)
        residuals.append(series[i] - blend)

    sigma2 = np.var(residuals) if residuals else 0.0
    bias_corrections[tname] = sigma2
    print(f"  {tname:<22}: σ² = {sigma2:.4f}  → correction factor = {np.exp(sigma2/2):.4f}")

# ============================================================
# 6. FINAL FORECAST — FULL HISTORY + BIAS CORRECTION
# ============================================================
print("\nSTEP 4: Generating Final Forecasts...")

full_preds_log = {}
for tname, series in TARGET_CONFIGS.items():
    w = final_weights[tname]
    method_preds = []
    for fn in MODEL_FNS:
        try:
            p = fn(series, H)
        except Exception:
            p = np.full(H, series[-1])
        method_preds.append(p)

    blend_log = np.dot(w, np.vstack(method_preds))

    # Apply log-normal bias correction
    sigma2 = bias_corrections[tname]
    corrected_log = blend_log + sigma2 / 2.0

    full_preds_log[tname] = corrected_log
    print(f"  ✓ {tname}")

freq_arr  = np.maximum(0, np.expm1(full_preds_log['Frequency_Log']))
total_arr = np.maximum(0, np.expm1(full_preds_log['Total_Claim_Log']))
sev_arr   = np.maximum(0, np.expm1(full_preds_log['Severity_Log']))

# Consistency blend: total from direct model vs freq×sev
# Weight toward whichever had better CV MAPE
freq_score  = cv_scores.get('Frequency_Log', 1)
sev_score   = cv_scores.get('Severity_Log', 1)
total_score = cv_scores.get('Total_Claim_Log', 1)
w_direct    = 1 / (total_score + 1e-8)
w_derived   = 1 / (freq_score + sev_score + 1e-8)
w_norm      = w_direct + w_derived
total_final = (w_direct / w_norm) * total_arr + (w_derived / w_norm) * (freq_arr * sev_arr)

# ============================================================
# 7. DAILY APPORTIONMENT + MONTHLY ROLL-UP
# ============================================================
print("STEP 5: Apportioning to Daily + Monthly Roll-up...")

daily_records = []
for i, week_end in enumerate(forecast_dates):
    week_start = week_end - pd.Timedelta(days=6)
    days = pd.date_range(start=week_start, end=week_end, freq='D')
    for d in days:
        dow = d.dayofweek
        daily_records.append({
            'Date':        d,
            'Daily_Freq':  freq_arr[i]    * dow_freq_w[dow],
            'Daily_Total': total_final[i] * dow_total_w[dow],
        })

daily_df = pd.DataFrame(daily_records)
daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')

monthly = daily_df.groupby('Month_Period').agg(
    Freq_Sum=('Daily_Freq', 'sum'),
    Total_Sum=('Daily_Total', 'sum'),
).reset_index()

final = monthly[
    (monthly['Month_Period'] >= pd.Period(FORECAST_START, freq='M')) &
    (monthly['Month_Period'] <= pd.Period(FORECAST_END, freq='M'))
].copy()

final['Frequency']   = np.round(final['Freq_Sum']).astype(int)
final['Total_Claim'] = final['Total_Sum']
final['Severity']    = final['Total_Claim'] / final['Frequency']

# ============================================================
# 8. EXPORT — EXACT SUBMISSION FORMAT
# ============================================================
rows = []
for _, row in final.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    rows.append({'id': f'{m_id}_Claim_Frequency', 'value': row['Frequency']})
    rows.append({'id': f'{m_id}_Claim_Severity',  'value': row['Severity']})
    rows.append({'id': f'{m_id}_Total_Claim',     'value': row['Total_Claim']})

submission_df = pd.DataFrame(rows)
submission_df.to_csv(OUTPUT_CSV, index=False)

# ============================================================
# 9. FINAL SUMMARY
# ============================================================
print("\n" + "=" * 62)
print("FINAL FORECAST — v5 MAPE-Optimized Ensemble")
print("=" * 62)
print(f"{'Month':<12} {'Frequency':>10} {'Severity':>20} {'Total Claim':>22}")
print("-" * 62)
for _, row in final.iterrows():
    print(f"{str(row['Month_Period']):<12} {row['Frequency']:>10,} "
          f"{row['Severity']:>20,.0f} {row['Total_Claim']:>22,.0f}")

print("\nCV MAPE per target (expanding-window):")
for tname, score in cv_scores.items():
    print(f"  {tname:<25}: {score:.2f}%")

print(f"\n✓ Saved → {OUTPUT_CSV}")
print("\nImprovements over v4 (5.49%):")
print("  [1] MAPE-optimized weights — matches actual evaluation metric")
print("  [2] Differential Evolution — true global weight optimum")
print("  [3] Log-normal bias correction: E[Y]=exp(μ+σ²/2) not exp(μ)")
print("  [4] 2 new diverse models: Naive Seasonal + Drift")
print("  [5] Expanding-window CV — more reliable weight estimation")
print("  [6] Per-target winsorization — adaptive outlier handling")
print("  [7] MAPE-weighted total consistency blend")

STEP 1: Data Preparation
  Weekly rows: 83
  Range: 2024-01-07 → 2025-08-03
  Winsorizing Frequency...
    Winsorized 4 values → [31, 83]
  Winsorizing Total_Claim...
    Winsorized 4 values → [988992570, 6615314152]
  Winsorizing Severity...
    Winsorized 4 values → [26629530, 85192850]
  Forecast horizon: 22 weeks

STEP 2: MAPE-Optimized Weight Search (Differential Evolution)...

  [Frequency_Log]
    DLinear        : MAPE = 10.58%
    ETS            : MAPE = 10.85%
    Theta          : MAPE = 10.17%
    SARIMA         : MAPE = 11.34%
    NaiveSeasonal  : MAPE = 16.94%
    Drift          : MAPE = 16.53%
    → Optimal weights: DLinear:0.00 ETS:0.27 Theta:0.73 SARIMA:0.00 NaiveSeasonal:0.00 Drift:0.00
    → Ensemble MAPE: 10.06%

  [Total_Claim_Log]
    DLinear        : MAPE = 30.25%
    ETS            : MAPE = 29.48%
    Theta          : MAPE = 25.86%
    SARIMA         : MAPE = 25.33%
    NaiveSeasonal  : MAPE = 51.70%
    Drift          : MAPE = 42.15%
    → Optimal weights: DLinea

In [12]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CORE INSIGHT: You're scored on MONTHLY outputs.
# Weekly → monthly means 20+ forecast steps and lots of noise.
# Monthly → monthly means only 5 steps on cleaner aggregates.
#
# With ~17 monthly rows, only ultra-lean models are safe:
#   [1] Theta          — M3 winner, 0 fittable params, robust
#   [2] SES            — 1 param (alpha), optimal for noisy data
#   [3] Holt Linear    — 2 params, captures trend
#   [4] DLinear(1 lag) — your original, adapted to monthly
#
# Weights: inverse-MAPE from Leave-One-Out CV (no holdout waste).
# No bias correction. No scipy. No complex features.
# ============================================================

FORECAST_MONTHS  = ['2025-08','2025-09','2025-10','2025-11','2025-12']
OUTPUT_CSV       = 'submission_v6_monthly.csv'
np.random.seed(42)

# ============================================================
# 1. DATA PREP — AGGREGATE TO MONTHLY
# ============================================================
print("=" * 60)
print("STEP 1: Monthly Aggregation")
print("=" * 60)

data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

monthly_df = valid_claims.resample('MS').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()
monthly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Month'}, inplace=True)
monthly_df['Severity'] = monthly_df['Total_Claim'] / monthly_df['Frequency'].replace(0, np.nan)

# Drop incomplete boundary months (partial data skews the series)
# Keep only months where we have a full calendar month of data
monthly_df = monthly_df.iloc[1:-1].reset_index(drop=True)  # drop first & last partial months

N = len(monthly_df)
print(f"  Monthly rows: {N}")
print(f"  Range: {monthly_df['Month'].min().strftime('%Y-%m')} → {monthly_df['Month'].max().strftime('%Y-%m')}")
print()
print(monthly_df[['Month','Frequency','Severity','Total_Claim']].to_string(index=False))

# Log transform
monthly_df['Freq_Log']  = np.log1p(monthly_df['Frequency'])
monthly_df['Total_Log'] = np.log1p(monthly_df['Total_Claim'])
monthly_df['Sev_Log']   = np.log1p(monthly_df['Severity'])

H = len(FORECAST_MONTHS)  # = 5 steps

# ============================================================
# 2. MODEL LIBRARY — LEAN, PROVEN ON SMALL SAMPLES
# ============================================================

def theta(series, H):
    """
    Theta Method. Zero fittable params on the series itself.
    Theta-0: OLS linear trend extrapolation (long-run)
    Theta-2: SES with optimal alpha (short-run)
    Forecast = 0.5 * Theta-0 + 0.5 * Theta-2
    """
    y = np.asarray(series, dtype=float)
    n = len(y)
    t = np.arange(1, n + 1)
    p = np.polyfit(t, y, 1)
    t_fut    = np.arange(n + 1, n + H + 1)
    theta0_f = np.polyval(p, t_fut)

    # Optimal SES alpha by minimising SSE
    best_a, best_sse = 0.1, np.inf
    for a in np.linspace(0.01, 0.99, 99):
        lvl, sse = y[0], 0.0
        for v in y[1:]:
            sse += (v - lvl) ** 2
            lvl  = a * v + (1 - a) * lvl
        if sse < best_sse:
            best_sse, best_a = sse, a

    lvl = y[0]
    for v in y[1:]:
        lvl = best_a * v + (1 - best_a) * lvl

    # SES forecast is constant from last level + linear drift correction
    theta2_f = np.array([lvl + p[0] * 0.5 * i for i in range(1, H + 1)])
    return 0.5 * theta0_f + 0.5 * theta2_f


def ses(series, H):
    """Simple Exponential Smoothing — 1 param, optimal for noisy short series."""
    try:
        m = ExponentialSmoothing(
            series, trend=None, seasonal=None,
            initialization_method='estimated'
        ).fit(optimized=True)
        return np.array(m.forecast(H))
    except Exception:
        return np.full(H, series[-1])


def holt(series, H):
    """Holt Linear — 2 params, captures trend without seasonal overfit."""
    try:
        m = ExponentialSmoothing(
            series, trend='add', seasonal=None,
            initialization_method='estimated'
        ).fit(optimized=True, remove_bias=True)
        return np.array(m.forecast(H))
    except Exception:
        return np.full(H, series[-1])


def dlinear_monthly(series, H, window=3, alpha=5.0, n_lags=3):
    """
    DLinear adapted for monthly: window=3 months, 3 lags, higher alpha.
    Rolling mean of 3 months ≈ quarterly trend — appropriate for monthly data.
    """
    s = pd.Series(series)
    df = pd.DataFrame({'y': s})
    df['trend'] = df['y'].rolling(window=window, min_periods=1).mean()
    df['resid'] = df['y'] - df['trend']
    for lag in range(1, n_lags + 1):
        df[f't{lag}'] = df['trend'].shift(lag)
        df[f'r{lag}'] = df['resid'].shift(lag)
    df = df.dropna()
    if len(df) < 5:
        return np.full(H, series[-1])
    t_f = [f't{i}' for i in range(1, n_lags + 1)]
    r_f = [f'r{i}' for i in range(1, n_lags + 1)]
    m_t = Ridge(alpha=alpha).fit(df[t_f], df['trend'])
    m_r = Ridge(alpha=alpha).fit(df[r_f], df['resid'])
    hist = list(series)
    preds = []
    for _ in range(H):
        s2   = pd.Series(hist)
        tr   = s2.rolling(window=window, min_periods=1).mean().values
        re   = (s2 - pd.Series(tr)).values
        pred = m_t.predict([[tr[-i] for i in range(1, n_lags+1)]])[0] \
             + m_r.predict([[re[-i] for i in range(1, n_lags+1)]])[0]
        preds.append(pred)
        hist.append(pred)
    return np.array(preds)


MODEL_NAMES = ['Theta', 'SES', 'Holt', 'DLinear']
MODEL_FNS   = [theta, ses, holt, dlinear_monthly]
M = len(MODEL_NAMES)

# ============================================================
# 3. LEAVE-ONE-OUT CV → INVERSE-MAPE WEIGHTS
#    LOO wastes no data (critical with only 17 rows).
#    Each model predicts the point it left out.
#    Weight = 1 / average_MAPE (better model → higher weight).
# ============================================================
print("\nSTEP 2: Leave-One-Out CV for Weights...")

TARGET_SERIES = {
    'Freq':  monthly_df['Freq_Log'].values,
    'Total': monthly_df['Total_Log'].values,
    'Sev':   monthly_df['Sev_Log'].values,
}

def loo_mape(series, model_fn):
    """LOO-CV: for each i >= min_train, train on all except i, predict i."""
    min_train = 6  # need at least 6 months to fit
    errors = []
    for i in range(min_train, len(series)):
        train = np.concatenate([series[:i]])   # all data before i
        try:
            pred_log = model_fn(train, 1)[0]
            pred     = np.expm1(pred_log)
            actual   = np.expm1(series[i])
            if actual > 0:
                errors.append(abs(pred - actual) / actual)
        except Exception:
            continue
    return np.mean(errors) if errors else 1.0

final_weights = {}

for tname, series in TARGET_SERIES.items():
    print(f"\n  [{tname}]")
    mapes = []
    for name, fn in zip(MODEL_NAMES, MODEL_FNS):
        m = loo_mape(series, fn)
        mapes.append(m)
        print(f"    {name:12s}: LOO-MAPE = {m*100:.2f}%")

    # Inverse-MAPE weights (better model → higher weight)
    inv = np.array([1.0 / (m + 1e-6) for m in mapes])
    w   = inv / inv.sum()
    final_weights[tname] = w
    print(f"    → Weights: " + " ".join(f"{n}:{v:.2f}" for n,v in zip(MODEL_NAMES, w)))

# ============================================================
# 4. FINAL FORECAST ON FULL HISTORY
# ============================================================
print("\nSTEP 3: Generating Forecasts...")

forecasts = {}
for tname, series in TARGET_SERIES.items():
    w = final_weights[tname]
    preds_all = []
    for fn in MODEL_FNS:
        try:
            p = fn(series, H)
        except Exception:
            p = np.full(H, series[-1])
        preds_all.append(p)

    blend_log = np.dot(w, np.vstack(preds_all))
    forecasts[tname] = np.maximum(0, np.expm1(blend_log))
    print(f"  ✓ {tname}")

freq_arr  = forecasts['Freq']
total_arr = forecasts['Total']
sev_arr   = forecasts['Sev']

# Final severity: use directly modeled severity (not derived)
# Final total: blend direct-modeled vs freq * sev
total_final = 0.5 * total_arr + 0.5 * (freq_arr * sev_arr)
sev_final   = np.where(freq_arr > 0, total_final / freq_arr, sev_arr)

# ============================================================
# 5. EXPORT — EXACT SUBMISSION FORMAT
# ============================================================
rows = []
for i, month_str in enumerate(FORECAST_MONTHS):
    freq  = max(1, round(freq_arr[i]))
    total = max(0, total_final[i])
    sev   = total / freq
    m_id  = month_str.replace('-', '_')
    rows.append({'id': f'{m_id}_Claim_Frequency', 'value': freq})
    rows.append({'id': f'{m_id}_Claim_Severity',  'value': sev})
    rows.append({'id': f'{m_id}_Total_Claim',     'value': total})

submission_df = pd.DataFrame(rows)
submission_df.to_csv(OUTPUT_CSV, index=False)

# ============================================================
# 6. RESULTS
# ============================================================
print("\n" + "=" * 60)
print("FINAL FORECAST — v6 Monthly Ensemble")
print("=" * 60)
print(f"{'Month':<12} {'Frequency':>10} {'Severity':>20} {'Total Claim':>22}")
print("-" * 60)
for i, month_str in enumerate(FORECAST_MONTHS):
    freq  = max(1, round(freq_arr[i]))
    total = total_final[i]
    sev   = total / freq
    print(f"{month_str:<12} {freq:>10,} {sev:>20,.0f} {total:>22,.0f}")

print("\n" + "=" * 60)
print(f"✓ Saved → {OUTPUT_CSV}")
print("\nWhy monthly modeling beats weekly for this problem:")
print("  • 5 forecast steps vs 20+ → 4x less error compounding")
print("  • Monthly aggregation smooths week-to-week noise")
print("  • LOO-CV uses ALL data for validation — no wasteful holdout")
print("  • Inverse-MAPE weights reward accuracy, not complexity")
print("  • No bias correction, no scipy, no overengineering")

STEP 1: Monthly Aggregation
  Monthly rows: 17
  Range: 2024-02 → 2025-06

     Month  Frequency     Severity  Total_Claim
2024-02-01        208 6.663291e+07 1.385965e+10
2024-03-01        278 5.147935e+07 1.431126e+10
2024-04-01        239 4.787056e+07 1.144106e+10
2024-05-01        263 4.643141e+07 1.221146e+10
2024-06-01        225 5.388963e+07 1.212517e+10
2024-07-01        257 5.825104e+07 1.497052e+10
2024-08-01        228 5.926726e+07 1.351294e+10
2024-09-01        208 5.896211e+07 1.226412e+10
2024-10-01        274 4.628163e+07 1.268117e+10
2024-11-01        270 5.086318e+07 1.373306e+10
2024-12-01        238 5.047861e+07 1.201391e+10
2025-01-01        216 4.449250e+07 9.610380e+09
2025-02-01        246 7.105911e+07 1.748054e+10
2025-03-01        230 5.947496e+07 1.367924e+10
2025-04-01        208 5.367427e+07 1.116425e+10
2025-05-01        239 5.115814e+07 1.222680e+10
2025-06-01        234 5.715008e+07 1.337312e+10

STEP 2: Leave-One-Out CV for Weights...

  [Freq]
    Theta 

In [13]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# v4 EXACT RESTORE — the 5.49% model, unchanged.
# The only addition: we also output an equal-weight variant
# (try submitting both and keep whichever scores lower).
# DO NOT ADD ANYTHING ELSE. This dataset is too small.
# ============================================================

FORECAST_START = '2025-08'
FORECAST_END   = '2025-12'
np.random.seed(42)

# ============================================================
# 1. DATA PREP
# ============================================================
data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])
valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

weekly_df = valid_claims.resample('W-SUN').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()
weekly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Week_End_Date'}, inplace=True)

# 3-sigma outlier cap (same as v4)
for col in ['Frequency', 'Total_Claim']:
    log_vals = np.log1p(weekly_df[col])
    mu, sigma = log_vals.mean(), log_vals.std()
    cap = np.expm1(mu + 3 * sigma)
    n_capped = (weekly_df[col] > cap).sum()
    weekly_df[col] = weekly_df[col].clip(upper=cap)
    if n_capped > 0:
        print(f"  Capped {n_capped} outlier weeks in {col}")

weekly_df['Freq_Log']  = np.log1p(weekly_df['Frequency'])
weekly_df['Total_Log'] = np.log1p(weekly_df['Total_Claim'])

N = len(weekly_df)
print(f"Weekly rows: {N}  |  {weekly_df['Week_End_Date'].min().date()} → {weekly_df['Week_End_Date'].max().date()}")

dow_freq  = valid_claims.groupby(valid_claims.index.dayofweek)['Nomor Polis'].count().reindex(range(7), fill_value=1)
dow_total = valid_claims.groupby(valid_claims.index.dayofweek)['Nominal Klaim Yang Disetujui'].sum().reindex(range(7), fill_value=1)
dow_freq_w  = (dow_freq  / dow_freq.sum()).values
dow_total_w = (dow_total / dow_total.sum()).values

last_date      = weekly_df['Week_End_Date'].max()
forecast_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=7),
    end=pd.to_datetime('2025-12-31') + pd.Timedelta(days=7),
    freq='W-SUN'
)
H = len(forecast_dates)

# ============================================================
# 2. MODELS — identical to v4
# ============================================================

def forecast_dlinear(series_log, H, window=4, alpha=1.0, n_lags=4):
    df = pd.DataFrame({'y': series_log})
    df['trend'] = df['y'].rolling(window=window, min_periods=1).mean()
    df['resid'] = df['y'] - df['trend']
    for lag in range(1, n_lags + 1):
        df[f't{lag}'] = df['trend'].shift(lag)
        df[f'r{lag}'] = df['resid'].shift(lag)
    df = df.dropna()
    t_f = [f't{i}' for i in range(1, n_lags + 1)]
    r_f = [f'r{i}' for i in range(1, n_lags + 1)]
    m_t = Ridge(alpha=alpha).fit(df[t_f], df['trend'])
    m_r = Ridge(alpha=alpha).fit(df[r_f], df['resid'])
    hist = list(series_log)
    preds = []
    for _ in range(H):
        s = pd.Series(hist)
        tr = s.rolling(window=window, min_periods=1).mean().values
        re = (s - pd.Series(tr)).values
        pred = m_t.predict([[tr[-i] for i in range(1, n_lags+1)]])[0] \
             + m_r.predict([[re[-i] for i in range(1, n_lags+1)]])[0]
        preds.append(pred)
        hist.append(pred)
    return np.array(preds)


def forecast_ets(series_log, H):
    best_aic, best_pred = np.inf, None
    for trend in ['add', None]:
        for seasonal in ['add', None]:
            try:
                sp = 4 if seasonal else None
                m = ExponentialSmoothing(
                    series_log, trend=trend, seasonal=seasonal,
                    seasonal_periods=sp, initialization_method='estimated'
                ).fit(optimized=True, remove_bias=True)
                if m.aic < best_aic:
                    best_aic, best_pred = m.aic, m.forecast(H)
            except Exception:
                continue
    if best_pred is None:
        m = ExponentialSmoothing(series_log, trend=None, seasonal=None).fit()
        best_pred = m.forecast(H)
    return np.array(best_pred)


def forecast_theta(series_log, H):
    y = np.array(series_log)
    n = len(y)
    t = np.arange(1, n + 1)
    p = np.polyfit(t, y, 1)
    t_fut = np.arange(n + 1, n + H + 1)
    theta0 = np.polyval(p, t_fut)
    best_alpha, best_sse = 0.1, np.inf
    for a in np.linspace(0.01, 0.99, 50):
        level, sse = y[0], 0.0
        for val in y[1:]:
            sse += (val - level) ** 2
            level = a * val + (1 - a) * level
        if sse < best_sse:
            best_sse, best_alpha = sse, a
    level = y[0]
    for val in y[1:]:
        level = best_alpha * val + (1 - best_alpha) * level
    theta2 = np.array([level + p[0] * 0.5 * i for i in range(1, H + 1)])
    return 0.5 * theta0 + 0.5 * theta2


def forecast_sarima(series_log, H):
    best_aic, best_pred = np.inf, None
    configs = [
        (1,1,1,0,0,0),(1,1,0,0,0,0),(0,1,1,0,0,0),
        (1,1,1,1,0,0),(1,1,1,0,1,0),(1,1,1,1,1,0),
        (0,1,1,1,1,0),(2,1,0,0,0,0),(0,1,2,0,0,0),
    ]
    for (p,d,q,P,D,Q) in configs:
        try:
            m = SARIMAX(series_log, order=(p,d,q),
                        seasonal_order=(P,D,Q,4),
                        enforce_stationarity=False,
                        enforce_invertibility=False).fit(disp=False)
            if m.aic < best_aic:
                best_aic, best_pred = m.aic, m.forecast(H)
        except Exception:
            continue
    return np.array(best_pred) if best_pred is not None else np.full(H, series_log[-1])


MODEL_FNS   = [forecast_dlinear, forecast_ets, forecast_theta, forecast_sarima]
MODEL_NAMES = ['DLinear', 'ETS', 'Theta', 'SARIMA']

target_logs = {
    'Frequency_Log':   weekly_df['Freq_Log'].values,
    'Total_Claim_Log': weekly_df['Total_Log'].values,
}

# ============================================================
# 3. WEIGHT SEARCH — identical to v4 (MAE, 8-week holdout, 0.1 grid)
# ============================================================
print("\nFinding optimal weights (v4 method)...")
VAL_WEEKS = 8
final_weights = {}

for tname, series in target_logs.items():
    train_s, val_s = series[:-VAL_WEEKS], series[-VAL_WEEKS:]
    preds = []
    for name, fn in zip(MODEL_NAMES, MODEL_FNS):
        try:
            p = fn(train_s, VAL_WEEKS)
            p = np.clip(p, series.min() - 1, series.max() + 1)
        except Exception:
            p = np.full(VAL_WEEKS, train_s[-1])
        mae = np.mean(np.abs(np.expm1(val_s) - np.expm1(p)))
        print(f"  {tname[:5]} {name:10s}: MAE={mae:,.0f}")
        preds.append(p)

    pm = np.vstack(preds)
    best_mae, best_w = np.inf, np.array([0.25]*4)
    for w0 in np.arange(0, 1.01, 0.1):
        for w1 in np.arange(0, 1.01-w0, 0.1):
            for w2 in np.arange(0, 1.01-w0-w1, 0.1):
                w3 = round(1.0 - w0 - w1 - w2, 8)
                if w3 < -0.001: continue
                w3 = max(0, w3)
                w  = np.array([w0, w1, w2, w3])
                if w.sum() < 0.99: continue
                blend = np.average(pm, axis=0, weights=w)
                mae   = np.mean(np.abs(np.expm1(val_s) - np.expm1(blend)))
                if mae < best_mae:
                    best_mae, best_w = mae, w.copy()

    final_weights[tname] = best_w
    print(f"  → {tname[:5]} weights: DLinear={best_w[0]:.1f} ETS={best_w[1]:.1f} Theta={best_w[2]:.1f} SARIMA={best_w[3]:.1f}\n")

# ============================================================
# 4. FORECAST — run twice: v4 weights AND equal weights
# ============================================================
def run_forecast(weights_dict, label):
    full_preds = {}
    for tname, series in target_logs.items():
        w = weights_dict[tname]
        mp = []
        for fn in MODEL_FNS:
            try: p = fn(series, H)
            except: p = np.full(H, series[-1])
            mp.append(p)
        blend = np.average(np.vstack(mp), axis=0, weights=w)
        full_preds[tname] = np.maximum(0, np.expm1(blend))

    freq_arr  = full_preds['Frequency_Log']
    total_arr = full_preds['Total_Claim_Log']
    sev_arr   = np.where(freq_arr > 0, total_arr / freq_arr, 0)

    weekly_preds = []
    for i, week_end in enumerate(forecast_dates):
        freq  = max(0, freq_arr[i])
        total = max(0, total_arr[i])
        sev   = max(0, sev_arr[i])
        total_final = 0.5 * total + 0.5 * (freq * sev)
        weekly_preds.append({
            'Week_Start_Date': week_end - pd.Timedelta(days=6),
            'Week_End_Date': week_end,
            'Frequency': freq, 'Total_Claim': total_final,
        })

    pred_df = pd.DataFrame(weekly_preds)
    daily_records = []
    for _, row in pred_df.iterrows():
        days = pd.date_range(start=row['Week_Start_Date'], end=row['Week_End_Date'], freq='D')
        for d in days:
            dow = d.dayofweek
            daily_records.append({
                'Date': d,
                'Daily_Freq':  row['Frequency']  * dow_freq_w[dow],
                'Daily_Total': row['Total_Claim'] * dow_total_w[dow],
            })

    daily_df = pd.DataFrame(daily_records)
    daily_df['Month_Period'] = daily_df['Date'].dt.to_period('M')
    monthly = daily_df.groupby('Month_Period').agg(
        Freq_Sum=('Daily_Freq','sum'), Total_Sum=('Daily_Total','sum')
    ).reset_index()

    final = monthly[
        (monthly['Month_Period'] >= pd.Period(FORECAST_START, freq='M')) &
        (monthly['Month_Period'] <= pd.Period(FORECAST_END, freq='M'))
    ].copy()
    final['Frequency']   = np.round(final['Freq_Sum']).astype(int)
    final['Total_Claim'] = final['Total_Sum']
    final['Severity']    = final['Total_Claim'] / final['Frequency']

    rows = []
    for _, row in final.iterrows():
        m_id = str(row['Month_Period']).replace('-', '_')
        rows.append({'id': f'{m_id}_Claim_Frequency', 'value': row['Frequency']})
        rows.append({'id': f'{m_id}_Claim_Severity',  'value': row['Severity']})
        rows.append({'id': f'{m_id}_Total_Claim',     'value': row['Total_Claim']})

    fname = f'submission_{label}.csv'
    pd.DataFrame(rows).to_csv(fname, index=False)

    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  {'Month':<12} {'Freq':>8} {'Severity':>18} {'Total':>18}")
    print(f"  {'-'*55}")
    for _, row in final.iterrows():
        print(f"  {str(row['Month_Period']):<12} {row['Frequency']:>8,} "
              f"{row['Severity']:>18,.0f} {row['Total_Claim']:>18,.0f}")
    print(f"\n  ✓ Saved → {fname}")
    return final

# Run v4 (CV-optimized weights)
run_forecast(final_weights, 'v4_restored')

# Run equal-weight variant (no weight overfitting)
equal_weights = {t: np.array([0.25, 0.25, 0.25, 0.25]) for t in target_logs}
run_forecast(equal_weights, 'v4_equal_weights')

print("\n" + "="*55)
print("SUBMIT BOTH AND KEEP THE LOWER MAPE.")
print("If equal weights wins → weight search was overfitting the 8-week window.")
print("If v4 weights wins   → CV was valid, keep v4 as final.")
print("="*55)

Weekly rows: 83  |  2024-01-07 → 2025-08-03

Finding optimal weights (v4 method)...
  Frequ DLinear   : MAE=6
  Frequ ETS       : MAE=6
  Frequ Theta     : MAE=7
  Frequ SARIMA    : MAE=6
  → Frequ weights: DLinear=0.0 ETS=1.0 Theta=0.0 SARIMA=0.0

  Total DLinear   : MAE=793,255,079
  Total ETS       : MAE=797,162,749
  Total Theta     : MAE=838,724,402
  Total SARIMA    : MAE=801,358,697
  → Total weights: DLinear=1.0 ETS=0.0 Theta=0.0 SARIMA=0.0


  v4_restored
  Month            Freq           Severity              Total
  -------------------------------------------------------
  2025-08           217         48,784,699     10,586,279,787
  2025-09           239         48,926,799     11,693,504,953
  2025-10           243         50,023,252     12,155,650,175
  2025-11           224         49,904,181     11,178,536,532
  2025-12           248         49,918,146     12,379,700,122

  ✓ Saved → submission_v4_restored.csv

  v4_equal_weights
  Month            Freq           Severit